# RSNA Knee: OAI track blend (private)

Track **OAI**: uses public weights trained with OAI external data.
Parts A+B: goodpjw2008's public 0.950 notebook unchanged. Part A2: nartaa 0.945 efficiency checkpoint, w=0.25. Part C: our R5t leg, w=0.1. Every added part falls back to the previous file on failure.


In [ ]:
def _own_figure():
    # Our contribution at a glance (drawn with matplotlib; numbers are from our own runs)
    import matplotlib.pyplot as plt
    from matplotlib.patches import FancyArrowPatch, FancyBboxPatch

    SURFACE, INK, INK2, MUTED, GRID, AXIS = "#fcfcfb", "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"
    BLUE, ORANGE, TINT, NEUTRAL = "#2a78d6", "#eb6834", "#cde2fb", "#f0efec"

    fig = plt.figure(figsize=(14, 8.6), facecolor=SURFACE)
    fig.text(0.03, 0.955, "What this notebook adds: our 2.5D reader next to the public 0.949 CoAtNet",
             fontsize=19, fontweight="bold", color=INK)
    fig.text(0.03, 0.918, "Part A is nartaa's single model (grey, unchanged). Part B is our ConvNeXt reader (blue), "
             "rank-blended in with one weight per finding.", fontsize=12.5, color=INK2)


    def box(ax, x, y, w, h, title, lines, fill):
        ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0,rounding_size=0.018", fc=fill,
                                    ec=(11 / 255, 11 / 255, 11 / 255, 0.10), lw=1))
        ax.text(x + w / 2, y + h - 0.09, title, ha="center", va="top", fontsize=12, fontweight="bold", color=INK)
        ax.text(x + w / 2, y + h - 0.27, lines, ha="center", va="top", fontsize=10.4, color=INK2, linespacing=1.45)


    def arrow(ax, x0, y0, x1, y1):
        ax.add_patch(FancyArrowPatch((x0, y0), (x1, y1), arrowstyle="-|>", mutation_scale=13, lw=1.4, color=MUTED,
                                     shrinkA=0, shrinkB=0))


    # ---- 1. our reader ----
    ax = fig.add_axes([0.03, 0.585, 0.94, 0.29])
    ax.set_xlim(0, 1), ax.set_ylim(0, 1), ax.axis("off")
    ax.text(0, 0.97, "1 · Our reader (Part B)", fontsize=13, fontweight="bold", color=INK, va="top")
    steps = [
        ("One MRI study", "up to 6 series:\n3 planes ×\nfat-sat / non-fat-sat", NEUTRAL),
        ("Canonical volumes", "0.4 mm per pixel\n154 mm field of view\nsame orientation,\nslices sorted by position", TINT),
        ("2.5D windows", "16 per series, spread\nover the stack;\n3 neighbouring slices\n= 3 channels", TINT),
        ("ConvNeXt-Tiny", "one shared 2D encoder\nfor every window\n(256 px, 3 folds)", TINT),
        ("Study transformer", "2 layers over all windows,\neach tagged with its\nseries type and\nslice position", TINT),
        ("12 attention pools", "one per finding:\neach looks at the\nwindows it needs", TINT),
    ]
    w, gap, h, y0 = 0.148, 0.0224, 0.74, 0.06
    for i, (t, s, f) in enumerate(steps):
        x = i * (w + gap)
        box(ax, x, y0, w, h, t, s, f)
        if i:
            arrow(ax, x - gap + 0.003, y0 + h / 2, x - 0.003, y0 + h / 2)

    # ---- 2. the blend ----
    ax = fig.add_axes([0.03, 0.04, 0.43, 0.50])
    ax.set_xlim(0, 1), ax.set_ylim(0, 1), ax.axis("off")
    ax.text(0, 0.98, "2 · The blend (public leaderboard, macro AUC)", fontsize=13, fontweight="bold", color=INK, va="top")


    def tile(x, y, w, h, label, value, note, fill, big=False):
        ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0,rounding_size=0.02", fc=fill,
                                    ec=(11 / 255, 11 / 255, 11 / 255, 0.10), lw=1))
        ax.text(x + 0.04, y + h - 0.045, label, fontsize=11, color=INK2, va="top")
        ax.text(x + 0.04, y + h * 0.46, value, fontsize=34 if big else 25, fontweight="bold", color=INK, va="center")
        ax.text(x + 0.04, y + 0.04, note, fontsize=10, color=INK2, va="bottom")


    tile(0.00, 0.52, 0.44, 0.33, "Part A · nartaa's CoAtNet", "0.949", "not ours · credited below", NEUTRAL)
    tile(0.00, 0.10, 0.44, 0.33, "Part B · our ConvNeXt reader alone", "0.929", "3 folds, trained only by us", TINT)
    tile(0.60, 0.26, 0.40, 0.43, "Rank blend, weight per finding", "0.950", "+0.001 over Part A alone", TINT, big=True)
    arrow(ax, 0.45, 0.685, 0.59, 0.53)
    arrow(ax, 0.45, 0.265, 0.59, 0.42)
    ax.text(0.0, 0.0, "A weaker but independently trained reader still helps, as long as\nits vote is small and it is allowed to speak only where it is strong.",
            fontsize=10.5, color=INK2, va="bottom", linespacing=1.4)

    # ---- 3. the weight each finding gives our reader ----
    names = ["Baker's cyst", "Contusion", "Medial OA", "ACL", "Medial meniscus", "MCL", "Lateral meniscus", "Fracture",
             "Lateral OA", "PF OA", "Effusion", "Synovitis"]
    wts = [0.35, 0.30, 0.25, 0.25, 0.20, 0.20, 0.08, 0.08, 0.08, 0.08, 0.04, 0.03]
    gold = [0.976, 0.966, 0.966, 0.965, 0.963, 0.946, 0.889, 0.911, 0.865, 0.843, 0.880, 0.761]
    ax = fig.add_axes([0.66, 0.075, 0.31, 0.36], facecolor=SURFACE)
    fig.text(0.50, 0.535, "3 · How much each finding listens to our reader", fontsize=13, fontweight="bold", color=INK, va="top")
    fig.text(0.50, 0.495, "bar: blend weight · label: our reader's AUC on the 58 radiologist-labelled studies", fontsize=10.5,
             color=INK2, va="top")
    ys = list(range(len(names)))[::-1]
    ax.barh(ys, wts, height=0.62, color=[BLUE if w_ >= 0.2 else AXIS for w_ in wts], zorder=2)
    for y, w_, g in zip(ys, wts, gold):
        ax.text(w_ + 0.008, y, f"{int(round(w_ * 100))}%  ·  AUC {g:.3f}", va="center", fontsize=10, color=INK)
    ax.set_yticks(ys), ax.set_yticklabels(names, fontsize=11, color=INK)
    ax.set_xlim(0, 0.62), ax.set_ylim(-0.7, len(names) - 0.3)
    ax.set_xticks([]), ax.tick_params(axis="y", length=0)
    for s in ax.spines.values():
        s.set_visible(False)
    plt.show()


try:
    _own_figure()
except Exception as _e:
    print('figure skipped:', repr(_e))
del _own_figure

**What this is.** The 2.5D ConvNeXt MIL reader that this notebook introduced on 4 October, now blended
with [nartaa's public 0.949 single CoAtNet](https://www.kaggle.com/code/nartaa/rsna-knee-0949-anatomical-mirror) instead of the community stack. One rank-blend weight
per finding: the reader gets up to 35% where it agrees well with the radiologists and 3 to 8% where it does not.
The public notebooks that use this recipe score **0.950**; this version's own run is being scored and the badge above will show it. Scoring takes about one to two hours instead of the stack's seven.

| | Public LB |
|---|---|
| nartaa's CoAtNet alone (Part A) | 0.949 |
| this notebook's reader alone, 3 folds | 0.929 |
| versions 1 to 5: community stack + reader at 30% | 0.944 |
| **this version: nartaa's CoAtNet + reader, per-finding weights** | **0.950** |

**Why it matters.** Every public notebook at 0.950 that we could find runs this reader (weights from
[goodpjw2008/rsna-knee-2-5d-convnext-reader](https://www.kaggle.com/datasets/goodpjw2008/rsna-knee-2-5d-convnext-reader)). This is the original, with the full source in
visible cells, the training code in the dataset, and the probes below that show what does and does not add to it.

**What we learned on the way** (our own public-LB submissions, 7 to 9 October 2026):

| On top of nartaa's 0.949 | Public LB |
|---|---|
| + the community 0.943 stack at 40% | 0.949 |
| + our readers (a fine-tuned Raptor x4 + this ConvNeXt x3) at 30% | 0.948 |
| + this reader x3 with per-finding weights, reader unchanged | 0.949 |
| + the same with two reader changes (this version) | 0.950 |

The weights alone do nothing; the two reader changes are what moved it: a missing fat-sat or non-fat-sat series
is filled with its counterpart, and 16 windows per series are read instead of 12. Filling a missing slot lowers the
reader's AUC against report-derived labels (0.885 to 0.878 on one fold) but raises it on the 58 radiologist-labelled
studies (0.908 to 0.914), and the test set is scored by radiologists.

## What is ours, and what is not

**Not ours.** Part A: nartaa's inference code and checkpoint, byte for byte (trained with OAI external data; see the
terms in [nartaa's dataset](https://www.kaggle.com/datasets/nartaa/rsna-knee-publication-swa-weights-20261007)). The per-finding weights and the two
reader changes come from [sujanmajhisuzan's Apex Grandmaster Stack](https://www.kaggle.com/code/sujanmajhisuzan/rsna-knee-apex-grandmaster-stack), which builds on this reader.

**Ours.** The reader itself: preprocessing (canonical volumes at 0.4 mm per pixel, orientation from
`ImageOrientationPatient`, slices sorted by position), the 2.5D model (ConvNeXt-Tiny over 3-slice windows, a two-layer
study transformer, one attention pool per finding), its training (five folds grouped by report text, soft labels
from four public label tables, the 58 gold studies never trained on), the three fold checkpoints, and the probes.

## Thanks

- **[nartaa (Danial Zakaria)](https://www.kaggle.com/nartaa)** for the 0.949 model and notebook that Part A is.
- **[sujanmajhisuzan](https://www.kaggle.com/sujanmajhisuzan)** for the per-finding weights and the reader changes.
- **[Dread Development](https://www.kaggle.com/dreaddevelopment)** for the Raptor recipe behind nartaa's model.
- **[skarin](https://www.kaggle.com/skarin)**, **[Mattia Angeli](https://www.kaggle.com/mattiaangeli)** and the
  authors of the community stack used in versions 1 to 5.
- **[stevenleehans](https://www.kaggle.com/stevenleehans)**, **[Pilkwang Kim](https://www.kaggle.com/pilkwang)**,
  **[lixin73](https://www.kaggle.com/lixin73)** and **[laymond](https://www.kaggle.com/laymond)** for the public
  report-label tables the reader was trained on.
- `timm` and the ConvNeXt and CoAtNet authors.

---
# Part A · nartaa's 0.949 single model (unchanged)

The next cells are [nartaa's notebook](https://www.kaggle.com/code/nartaa/rsna-knee-0949-anatomical-mirror) as published.

# RSNA Knee: accuracy inference (0.949 public AUC)

**Runnable inference notebook.** The exact checkpoint is attached and a T4 GPU is enabled. This version runs the scored pipeline and writes `submission.csv`.

Original 384-pixel full-field training; native 320-pixel center crop at inference, K94, two views with equal plain/anatomical-mirror probability averaging. One seed-42 training run, one averaged checkpoint at inference; averaged epochs [14, 11, 12]. The accuracy model is the original 384-trained checkpoint: the later native-320-crop-trained alternative scored 0.946 and is not substituted here.

The score is the displayed ordinary full-12 public leaderboard macro-AUC for submission `56906959`. The observed 26.1 minutes is submit-to-scored turnaround including queueing, orchestration and polling, not official worker runtime or an efficiency rank. Gold58 was reused for selection and development; it is not independent CV. This release does not establish significance or private-test performance.

The scored source's opening docstring and some comments describe older Raptor versions (0.924, 64 slices, 42 windows and earlier ensembles). They are retained to preserve the complete scored source bytes. This introduction, the live constants and the manifest describe the selected recipe: 96 slices, 94 windows, and one checkpoint.


## Reproduction requirements

Accept the competition rules and attach `rsna-knee-abnormality-detection`. Attach the exact checkpoint `raptor_ft_alldata_t16_blendjev_oai_d96_r384_swa.pt` from the [selected SWA weights](https://www.kaggle.com/datasets/nartaa/rsna-knee-publication-swa-weights-20261007), version 1. This public dependency contains only learned checkpoints and aggregate metadata. OAI images, OAI caches, report text and training label tables are not inference inputs and are not distributed.

Use the original Kaggle GPU environment: two Tesla T4 GPUs were available in the scored machine; the unchanged source uses its original CUDA worker policy. Runtime dependencies are Python, PyTorch with CUDA, timm, NumPy, pandas, pydicom and OpenCV. No downloads or pip installs are added. The scored Kaggle container is pinned to `gcr.io/kaggle-private-byod/python@sha256:2757e0c7d1e0a9cb43da657b97e223c321a98f5014bdf64f44f2f6b083ad2b2f`. The associated Kaggle qualification recorded torch 2.11.0+cu128 and timm 1.0.29.

Source SHA-256: `bcd864cbf349882e876959b49cb348cbb89b4c62fc204d4e4261049c5e3a79c5`. Checkpoint SHA-256: `7e5315dad125b99fc65b340b3de41de628e9be51ff5835355dd61c86472244ef`. The original private scored source is `nartaa/rsna-knee-original384-s42-mirror320-infer`, version 1. Its private dependency is `nartaa/rsna-knee-oai-original384-s42-pair`, version 1; it remains private. This publication uses a separate notebook slug and leaves the private scored notebook unchanged.

The metadata enables T4 GPUs with internet disabled. `RUN_INFERENCE = True` runs the two code cells in order after verifying the attached checkpoint. The final source writes `submission.csv`; competition submission is managed separately.


## Credits and required acknowledgement

Built on dreaddevelopment's [Raptor CoAtNet/MIL inference recipe and public baseline weights](https://www.kaggle.com/datasets/dreaddevelopment/raptor-knee-widedense). These are our retrained checkpoints, rather than those baseline weights. Weak-label sources include [stevenleehans](https://www.kaggle.com/datasets/stevenleehans/rsna-knee-llm-report-labels), [pilkwang](https://www.kaggle.com/datasets/pilkwang/rsna-knee-llm-labels), and [riadmohamed42's JEV labels and verification](https://www.kaggle.com/code/riadmohamed42/jev-knee-labels-verification), combined with our report-derived labels. Source reports and study-level label tables are not included here. The CoAtNet backbone comes from timm. Competition data is provided under the [RSNA Knee competition rules](https://www.kaggle.com/competitions/rsna-knee-abnormality-detection/rules).

Data and/or research tools used in the preparation of this manuscript were obtained and analyzed from the controlled access datasets distributed from the Osteoarthritis Initiative (OAI), a data repository housed within the NIMH Data Archive (NDA). OAI is a collaborative informatics system created by the National Institute of Mental Health and the National Institute of Arthritis, Musculoskeletal and Skin Diseases (NIAMS) to provide a worldwide resource to quicken the pace of biomarker identification, scientific investigation and OA drug development. Dataset identifier(s): 10.15154/0hcg-f676.

[Shared NDA Study](https://nda.nih.gov/study.html?id=3407) · [DOI: 10.15154/0hcg-f676](https://doi.org/10.15154/0hcg-f676). Official acknowledgement source: [NDA manuscript preparation](https://nda.nih.gov/nda/manuscript-preparation).


In [ ]:
# Authorized scoring run: verify the selected checkpoint before inference.
RUN_INFERENCE = True
if not RUN_INFERENCE:
    raise SystemExit("Static notebook only. Attach the exact checkpoint and competition, enable a T4 GPU, and set RUN_INFERENCE = True to run inference.")

import hashlib
from pathlib import Path
import torch
if not torch.cuda.is_available():
    raise RuntimeError("This scored pipeline requires CUDA; CPU inference is not enabled.")
_weight_name = 'raptor_ft_alldata_t16_blendjev_oai_d96_r384_swa.pt'
_candidates = [
    Path('/kaggle/input/rsna-knee-publication-swa-weights-20261007') / _weight_name,
    Path('/kaggle/input/datasets/nartaa/rsna-knee-publication-swa-weights-20261007') / _weight_name,
    Path('/kaggle/input/rsna-knee-oai-original384-s42-pair') / _weight_name,
    Path('/kaggle/input/datasets/nartaa/rsna-knee-oai-original384-s42-pair') / _weight_name,
]
_weights = next((p for p in _candidates if p.is_file()), None)
if _weights is None:
    raise FileNotFoundError("Attach the exact selected weights; no alternative model is substituted.")
with _weights.open('rb') as _f:
    assert hashlib.file_digest(_f, 'sha256').hexdigest() == '7e5315dad125b99fc65b340b3de41de628e9be51ff5835355dd61c86472244ef', "Checkpoint hash differs"


In [ ]:
# %% [code]
#!/usr/bin/env python3
"""Knee MRI: twelve findings from a single model

This notebook takes a knee MRI study and scores twelve findings at once: ACL tear, MCL tear,
medial and lateral meniscus tears, osteoarthritis in the medial, lateral and patellofemoral
compartments, joint effusion, synovitis, a Baker's cyst, bone contusion and fracture. It scores
0.924 on the public leaderboard using one model, with no ensembling and no test-time augmentation.

This is the inference half of the work. The model was trained separately and its weights are
attached as a dataset, so this notebook only loads them and predicts:
https://www.kaggle.com/datasets/dreaddevelopment/raptor-knee-widedense

Where the training labels came from

Worth saying up front, because it shapes everything else. The competition gives you 4,407 studies
but structured labels for only 58 of them. Every other study arrives with a free-text radiology
report and nothing more, so there is very little to train against out of the box.

The labels behind these weights were made by reading those reports with a language model and
turning each into twelve probabilities rather than twelve yes or no answers. A report that says a
tear is suspected becomes a number near 0.8, not a 1, which is a fairer target than forcing every
hedged sentence into a hard label. That yields 4,349 studies to train on. The 58 studies that came
with real labels were never trained on and are used to check the result honestly; the model reaches
0.9167 macro-AUC on them.

Building a fixed input from studies that are all shaped differently

The hard part of this competition is not the network, it is that no two studies look alike. A
study holds several DICOM series shot in different planes, the number of series varies, and the
number of slices in a series varies more. Anything that expects a fixed-size input has to be given
one.

The approach here is to fill five fixed slots per study, always in the same order, for a stack of
64 images:

  18 slices from a sagittal series, preferring a fluid-sensitive one
  14 slices from a second sagittal series, preferring one that is not fluid-sensitive
  12 slices from a coronal series, preferring a fluid-sensitive one
   8 slices from a second coronal series
  12 slices from an axial series

Preferring a fluid-sensitive series for some slots and not for others is deliberate. Fluid-
sensitive sequences show swelling, effusion and acute injury clearly, while the other sequences
show anatomy and cartilage better, and the twelve findings are split across both. If a study has
no series for a slot, the slot is left as zeros and the model is told to skip it rather than being
fed something misleading.

Within a series, slices are taken evenly across 6 to 94 percent of the stack rather than from the
middle. The outer slices are where the collateral ligaments and the lateral meniscus sit, and
cutting them was measurably costing accuracy on exactly those findings.

Every slice is cropped to a 140 mm box around the centre of the image using the pixel spacing from
the DICOM header, then resized to 336 pixels. Cropping by millimetres rather than by pixel count
matters: it means a knee occupies the same fraction of the frame whether the scan was acquired at
0.3 or 0.5 mm per pixel, so the model is not asked to learn scale differences that carry no medical
information.

How the model reads the stack

Three neighbouring slices are stacked into the three channels of one image. The network then sees
a little of what lies above and below the slice in the middle, which is most of the benefit of a 3D
model at the cost of a 2D one. Each of these three-slice windows is passed through a CoAtNet
backbone at 384 pixels.

The windows are combined with an attention layer that has separate weights for each of the twelve
findings. This is the part that matters most. A cruciate tear may be visible on two sagittal slices
while osteoarthritis is spread across many coronal ones, and a single pooled score forces those two
to share one notion of which slices are important. Giving each finding its own attention weights
lets each one draw on the slices that actually show it.

Running it

Scoring uses 42 windows per study. Inference runs in half precision and automatically retries a
study in full precision if it fails, so no study is ever dropped from the submission. The notebook
needs no internet: the backbone is loaded from the attached weights rather than downloaded.
"""
import os, sys, glob, time, json, gc
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")
os.environ.setdefault("HF_HUB_DISABLE_TELEMETRY", "1")
import numpy as np
from concurrent.futures import ThreadPoolExecutor
import threading, queue
import torch, torch.nn as nn, torch.nn.functional as F
import timm
# T4 (Turing) cuDNN v9 has fp16/fp32 conv engines but NOT bf16 for these shapes
# ("GET was unable to find an engine..."); benchmark lets it pick a valid algo for
# the fixed (1,24,3,res,res) input.
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True

# ---- fixed config (must match training exactly) -----------------------------
INPUT_RES = 320
INPUT_MODE = "crop"
IMG = 384   # native 384 corpus: the model saw 384 px crops directly, no 336->384 upsample
CROP_MM = 140.0
# 96 slices per study at native 384 px. Must match the corpus the weights were trained on
# (cache/build_corpus96.py).
SLOTS = [("Sagittal", 1, 26), ("Sagittal", 0, 22), ("Coronal", 1, 18),
         ("Coronal", 0, 12), ("Axial", -1, 18)]
MAXS = sum(s[2] for s in SLOTS)                     # 96
K_EVAL = 94   # every window position of the 96-slice test volume (full coverage)
HDR_THREADS = 16   # per prep process: header reads are fuse-latency-bound, not CPU-bound
PX_THREADS = 8     # per prep process: pixel reads of the ~96 picked slices
QUEUE_MAX = 48     # studies in flight between the prep pool and the GPU consumers (RAM bound: ~14 MB each)
NORM = "imagenet"
FASTREAD_VARIANT = 'study_lut'   # kernel v3: research/fastread/readers.py, byte-identical to build_study
FR_HDR, FR_PX = 32, 16
_FASTREAD_B64 = "..."
def _load_fastread():
    import base64 as _b, importlib.util as _iu
    p = '/kaggle/working/_fastread.py' if os.path.isdir('/kaggle/working') else os.path.join(os.path.dirname(os.path.abspath(__file__)), '_fastread.py')
    if not os.path.exists(p):
        open(p, 'wb').write(_b.b64decode(_FASTREAD_B64))
    spec = _iu.spec_from_file_location('_fastread', p); m = _iu.module_from_spec(spec); spec.loader.exec_module(m)
    return m
LAB = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA",
       "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]
_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
_STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

# Three arms: (weights filename, fallback arch, fallback res). ck carries arch+res too.
# Selected 2026-08-19 by greedy forward selection AND exhaustive subset search over a 7-arm
# panel on the 45-study gold set (phase2/blend_panel.py); both agree on this exact set.
# Singles: coatnet384 0.9025 | swinbase384 0.8825 | effv2l480 0.8716.
# Blend {coatnet+swin+effv2l} = 0.9068 (2-arm {coatnet+swin} = 0.9059, coatnet alone 0.9025).
# Dropped as redundant: cnn336 (0.8833, the former champion), cnbase384 (0.8754),
# cnlarge384 (0.8752), maxvit384 (0.8438).
#
# SINGLE ARM: coatnet_rmlp_2_rw_384 retrained on the EXPANDED 4,349-study corpus.
#
# Why one arm and not the 3-arm blend: on the live leaderboard CoAtNet alone scored 0.914 while
# every blend scored 0.914-0.915, so ensembling is worth ~+0.001 there -- the ~+0.010 it showed
# on the old 45-study gold set was gold-set noise. One arm is also 1/3 the kernel runtime.
#
# Corpus expansion: the corpus previously held 3,200 of the 4,349 labelled studies and only 45
# of the 58 gold studies. Rebuilt to 4,407 studies (+37.8% training data, 58-study gate).
#
# Measured on the 58-study gate (the incumbent re-scored on the SAME gate for a fair compare):
#   incumbent CoAtNet (3,155-study corpus) 0.8923
#   this model       (4,349-study corpus) 0.9054   (+0.0131, better in 92.7% of 2000 bootstraps)
# Biggest gains land on the findings that were capping us: Lateral Meniscus +0.071,
# Fracture +0.057, Lateral OA +0.048, Medial Meniscus +0.035, ACL +0.028.
ARMS = [   # our 5-fold CoAtNet (public recipe reproduced on our folds/labels), equal-weight rank-mean
    {"file": "raptor_ft_alldata_t16_blendjev_oai_d96_r384_swa.pt", "arch": "coatnet_rmlp_2_rw_384.sw_in12k_ft_in1k", "res": 384, "w": 1.0},
]


# ============================================================================
# Model -- verbatim from finetune_raptor.py
# ============================================================================
def build_backbone(arch, pretrained=False):
    # maxvit/maxxvit/coatnet are conv-attention hybrids: NO CLS token, NO interpolatable
    # pos-embed -> avg pool. The "vit" substring in "coatnet"/"maxvit" must NOT route them
    # down the ViT path (mirrors finetune_raptor.py exactly).
    hybrid = arch.startswith(("maxvit", "maxxvit", "coatnet", "coat_", "convnext"))
    is_vit = (not hybrid) and any(k in arch for k in ("vit", "deit", "dinov2", "eva", "beit"))
    kw = dict(pretrained=pretrained, num_classes=0, in_chans=3, img_size=INPUT_RES)
    if is_vit:
        kw.update(global_pool="token", dynamic_img_size=True)
    else:
        kw.update(global_pool="avg")
    return timm.create_model(arch, **kw)


class RaptorClassifier(nn.Module):
    def __init__(self, backbone, F_dim=768, n=12, drop=0.2):
        super().__init__()
        self.backbone = backbone
        self.norm = nn.LayerNorm(F_dim)
        self.att = nn.Sequential(nn.Linear(F_dim, 256), nn.Tanh(), nn.Dropout(drop),
                                 nn.Linear(256, n))
        self.clsW = nn.Parameter(torch.zeros(n, F_dim))
        self.clsb = nn.Parameter(torch.zeros(n))
        nn.init.trunc_normal_(self.clsW, std=0.02)
        self.n = n

    def encode(self, x):
        B, K = x.shape[:2]
        # Bound backbone memory; the attention head still sees every window together.
        flat = x.flatten(0, 1)
        f = torch.cat([self.backbone(chunk) for chunk in flat.split(16)], dim=0)
        return f.view(B, K, -1)

    def head(self, feats):
        h = self.norm(feats)
        a = self.att(h)
        a = torch.softmax(a, dim=1)
        pooled = torch.einsum("bkn,bkf->bnf", a, h)
        logits = (pooled * self.clsW).sum(-1) + self.clsb
        return logits

    def forward(self, x):
        return self.head(self.encode(x))


def load_model(pt_path, arch_default, res_default, device, ngpu=1):
    import hashlib
    with open(pt_path, "rb") as stream:
        observed_sha = hashlib.file_digest(stream, "sha256").hexdigest()
    assert observed_sha == '7e5315dad125b99fc65b340b3de41de628e9be51ff5835355dd61c86472244ef', "Bound checkpoint bytes differ"
    print(f"[weights] sha256 {observed_sha}", flush=True)
    ck = torch.load(pt_path, map_location="cpu", weights_only=False)
    arch = ck.get("arch", arch_default)
    assert int(ck.get("res", res_default)) == 384, "Expected original 384 checkpoint"
    ck_res = INPUT_RES
    bb = build_backbone(arch, pretrained=False)
    model = RaptorClassifier(bb, F_dim=bb.num_features)
    model.load_state_dict(ck["model"], strict=True)
    model.eval().to(device)
    # NOTE: DataParallel removed on purpose. On the full hidden test it drove a system-RAM OOM
    # (per-forward module replication over many studies); a single T4 handles K_EVAL=24 windows
    # fine. Arms are also run SEQUENTIALLY (see main) so peak RAM == one model, not two.
    del ck
    gc.collect()
    return model, ck_res


# ============================================================================
# Eval windowing -- verbatim from finetune_raptor.py StudyWindows (train=False)
# ============================================================================
def _eval_centers(mask, D, k):
    valid = np.where(mask > 0)[0]
    if len(valid) < 3:
        valid = np.arange(min(3, D))
    lo, hi = int(valid.min()), int(valid.max())
    cs = [c for c in range(lo + 1, hi) if c - 1 >= lo and c + 1 <= hi]
    if not cs:
        cs = [max(1, min((lo + hi) // 2, D - 2))]
    idx = np.linspace(0, len(cs) - 1, k).round().astype(int)
    return [cs[i] for i in idx]


SLOT_SIZES = [26, 22, 18, 12, 18]
SAGITTAL_SLOTS = (0, 1)

def _slot_index(D):
    b = np.cumsum([0] + SLOT_SIZES)
    if b[-1] != D: raise RuntimeError(f'slot sizes sum to {b[-1]}, volume has {D} slices')
    s_ = np.searchsorted(b, np.arange(D), side='right') - 1
    rel = (np.arange(D) - b[s_]) / np.maximum(1, np.array(SLOT_SIZES)[s_] - 1)
    return s_, rel

def _mirror_tri(tri, cs, D):
    """final-g1-bundle MIRROR_TTA: true L/R mirror of (k,3,H,W) uint8 eval triplets (research/rnd4/mirror/mirror_f0.py). The slot of each
    window's CENTRE slice decides the axis: sagittal (slots 0,1) -> reverse the 3 slice-channels; coronal/axial (2,3,4) -> flip the width.
    Done on uint8 BEFORE the per-channel LUT, i.e. 'mirror the knee, then normalise'."""
    s_, _ = _slot_index(D)
    sag = torch.from_numpy(np.isin(s_[np.asarray(cs, dtype=np.int64)], SAGITTAL_SLOTS)).to(tri.device).view(-1, 1, 1, 1)
    return torch.where(sag, tri.flip(1), tri.flip(-1))

def infer_pair_probs(model, plain, mirrored, device):
    assert plain.shape==mirrored.shape==(94,3,320,320)
    assert all(not m.training for m in model.modules()), "TTA must not update BN or enable dropout"
    p = infer_probs(model, plain, device)
    q = infer_probs(model, mirrored, device)
    return 0.5 * (p + q)

def eval_windows(vol, mask, k, res, norm=NORM, mirror=False):
    D = vol.shape[0]
    cs = _eval_centers(mask, D, k)
    wins = np.empty((len(cs), 3, res, res), np.float32)
    for j, c in enumerate(cs):
        c = max(1, min(c, D - 2))
        tri = np.stack([vol[c - 1], vol[c], vol[c + 1]], 0)
        if mirror:
            tri = _mirror_tri(torch.from_numpy(tri)[None], [c], D)[0].numpy()
        tri = tri.astype(np.float32) / 255.0
        assert tri.shape[-2:] == (IMG, IMG) and res == INPUT_RES
        lo = (IMG - res) // 2
        tri = tri[:, lo:lo + res, lo:lo + res]
        t = torch.from_numpy(np.ascontiguousarray(tri))
        if t.shape[-1] != res:
            t = F.interpolate(t[None], size=(res, res), mode="bilinear",
                              align_corners=False)[0]
        wins[j] = t.numpy()
    x = torch.from_numpy(wins)
    if norm == "imagenet":
        x = (x - _MEAN) / _STD
    return x


_WLUT = {}
def gpu_windows(vol, mask, k, res, dev, norm=NORM, mirror=False):
    """Same windows as eval_windows, built on the GPU from the uint8 volume. Values come from a per-channel
    256-entry table computed on the CPU with eval_windows' own float32 ops, so they are bit-identical."""
    D = vol.shape[0]
    cs = np.array([max(1, min(c, D - 2)) for c in _eval_centers(mask, D, k)])
    tri = torch.from_numpy(np.ascontiguousarray(vol[np.stack([cs - 1, cs, cs + 1], 1)])).to(dev)   # (k,3,IMG,IMG) uint8
    if mirror:
        tri = _mirror_tri(tri, cs, D)
    assert tri.shape[-2:] == (IMG, IMG) and res == INPUT_RES
    lo = (IMG - res) // 2
    tri = tri[:, :, lo:lo + res, lo:lo + res]
    key = (str(dev), norm)
    if key not in _WLUT:
        v = torch.from_numpy((np.arange(256, dtype=np.uint8).astype(np.float32) / 255.0))
        t = torch.stack([v, v, v], 0).view(3, 256, 1)
        if norm == 'imagenet':
            t = (t - _MEAN.view(3, 1, 1)) / _STD.view(3, 1, 1)
        _WLUT[key] = t.view(3, 256).contiguous().to(dev)
    L = _WLUT[key]
    x = torch.stack([L[c][tri[:, c].long()] for c in range(3)], 1)                   # exact float32 values
    if x.shape[-1] != res:
        if norm == 'imagenet':   # resize must happen BEFORE normalisation, as in eval_windows
            L0 = _WLUT.setdefault((str(dev), 'none'), torch.from_numpy(np.arange(256, dtype=np.uint8).astype(np.float32) / 255.0).to(dev))
            x = L0[tri.long()]
            x = F.interpolate(x, size=(res, res), mode='bilinear', align_corners=False)
            x = (x - _MEAN.to(dev)) / _STD.to(dev)
        else:
            x = F.interpolate(x, size=(res, res), mode='bilinear', align_corners=False)
    return x


@torch.no_grad()
def infer_probs(model, xwins, device):
    assert str(device).startswith("cuda"), "Qualified inference requires CUDA/fp16"
    x = xwins.unsqueeze(0).to(device)
    with torch.autocast("cuda", dtype=torch.float16):
        logits = model(x).float()
        assert torch.isfinite(logits).all(), "Nonfinite fp16 logits"
        o = torch.sigmoid(logits)
    return o[0].cpu().numpy()


def rankpct(x):                                   # per-column percentile rank in [0,1]
    order = x.argsort(0).argsort(0).astype(np.float64)
    return order / max(1, (x.shape[0] - 1))


# ============================================================================
# Preprocessing -- verbatim from kprep2/dino_preprocess.py, retargeted to TEST
# ============================================================================
def _make_reader():
    import pydicom, cv2
    from pydicom.pixel_data_handlers.util import apply_modality_lut

    def _hdr(f):
        try:
            h = pydicom.dcmread(f, stop_before_pixels=True)
            iop = getattr(h, 'ImageOrientationPatient', None)
            ipp = getattr(h, 'ImagePositionPatient', None)
            if iop is not None and ipp is not None and len(iop) == 6:
                r = np.array(iop[:3], float); c = np.array(iop[3:], float)
                n = np.cross(r, c); pos = float(np.dot(np.array(ipp, float), n))
            else:
                pos = float(getattr(h, 'InstanceNumber', 0) or 0)
            ps = getattr(h, 'PixelSpacing', None); ps = float(ps[0]) if ps is not None else 0.5
            return (pos, f, ps, True)
        except Exception:
            return (0.0, f, 0.5, False)

    def order_and_meta(sdir):
        # Header reads are latency-bound on Kaggle's mounted competition data (~240k opens on the hidden
        # set); a thread pool hides that latency. Records and sort are unchanged -> identical ordering.
        fs = glob.glob(sdir + "/*.dcm")
        with ThreadPoolExecutor(max_workers=HDR_THREADS) as ex:
            out = list(ex.map(_hdr, fs))
        recs = [(pos, f, ps) for pos, f, ps, _ in out]; ps_list = [ps for _, _, ps, ok in out if ok]
        recs.sort(key=lambda x: x[0])
        med_ps = float(np.median(ps_list)) if ps_list else 0.5
        return [(f, ps) for _, f, ps in recs], med_ps

    def read_px(f):
        d = pydicom.dcmread(f)
        a = apply_modality_lut(d.pixel_array, d).astype(np.float32)
        if str(getattr(d, 'PhotometricInterpretation', '')) == 'MONOCHROME1':
            a = a.max() - a
        return a

    def mm_crop_resize(a, ps):
        h, w = a.shape; cpx = int(round(CROP_MM / max(ps, 1e-3)))
        cpx = min(cpx, min(h, w)); y0 = (h - cpx) // 2; x0 = (w - cpx) // 2
        a = a[y0:y0 + cpx, x0:x0 + cpx]
        return cv2.resize(a, (IMG, IMG), interpolation=cv2.INTER_AREA)

    return order_and_meta, read_px, mm_crop_resize


def _pick_series_for_slot(rows, plane, fluid, used):
    cands = [r for r in rows if r['Anatomical_Plane'] == plane and r['SeriesInstanceUID'] not in used]
    if fluid in (0, 1):
        pref = [r for r in cands if int(r.get('Fluid_Sensitive', 0) or 0) == fluid]
        if pref:
            return pref[0]
    return cands[0] if cands else None


def build_study(sid, ser_records, tsdir, reader):
    order_and_meta, read_px, mm_crop_resize = reader
    rows = ser_records.get(sid, [])
    vol = np.zeros((MAXS, IMG, IMG), np.uint8); idx = 0; used = set()
    for plane, fluid, k in SLOTS:
        r = _pick_series_for_slot(rows, plane, fluid, used)
        if r is None:
            idx += k; continue
        used.add(r['SeriesInstanceUID'])
        files, med_ps = order_and_meta(f"{tsdir}/{sid}/{r['SeriesInstanceUID']}")
        if not files:
            idx += k; continue
        # wide span: the collateral ligaments and lateral meniscus live in the
        # peripheral slices the old 0.15-0.85 crop threw away. Must match the corpus
        # the weights were trained on (build_corpus80.py, SPAN).
        n = len(files); lo, hi = int(n * 0.02), int(n * 0.98) - 1; hi = max(hi, lo)
        picks = np.linspace(lo, hi, k).round().astype(int) if n > 1 else [0] * k
        sel = [files[min(p, n - 1)] for p in picks]
        def _px(fp_ps):
            fp, ps = fp_ps
            try: return read_px(fp), ps
            except Exception: return None, med_ps
        with ThreadPoolExecutor(max_workers=PX_THREADS) as ex:
            got = list(ex.map(_px, sel))                    # order preserved
        arrs = [a for a, _ in got]; pss = [ps for _, ps in got]
        valid = [a for a in arrs if a is not None]
        if valid:
            allpx = np.concatenate([a.ravel() for a in valid])
            loq, hiq = np.percentile(allpx, [2.0, 98.0])
        else:
            loq, hiq = 0.0, 1.0
        for a, ps in zip(arrs, pss):
            if idx >= MAXS: break
            if a is None: idx += 1; continue
            aw = np.clip((a - loq) / (hiq - loq + 1e-6), 0, 1)
            aw = mm_crop_resize(aw, ps if ps > 0 else med_ps)
            vol[idx] = (aw * 255).astype(np.uint8); idx += 1
        if idx >= MAXS: break
    mask = (vol.reshape(MAXS, -1).sum(1) > 0).astype(np.uint8)
    return vol, mask



# ---- parallel preprocessing helpers (module level so they pickle under fork) ----
_SER = None; _TSDIR = None; _READER = None
def _pre_init():
    global _READER
    _READER = _load_fastread().make_builder(FASTREAD_VARIANT, IMG, SLOTS, hdr_threads=FR_HDR, px_threads=FR_PX)
def _pre_one(sid):
    try:
        vol, mask = _READER(sid, _SER, _TSDIR)
    except Exception as e:
        print(f"  [pre] {sid[:16]} FAILED ({type(e).__name__}: {e})", flush=True)
        vol = np.zeros((MAXS, IMG, IMG), np.uint8); mask = np.zeros(MAXS, np.uint8)
    return sid, vol, mask

# ============================================================================
# Test-root discovery + weights + main
# ============================================================================
def find_test_root():
    cands = ["/kaggle/input/competitions/rsna-knee-abnormality-detection",
             "/kaggle/input/rsna-knee-abnormality-detection"]
    for b in cands:
        if os.path.exists(b + "/test.csv"):
            return b
    for d, _, f in os.walk("/kaggle/input"):
        if "test.csv" in f and (os.path.isdir(d + "/test_series") or os.path.isdir(d + "/test_images")):
            return d
    for d, _, f in os.walk("/kaggle/input"):
        if "test.csv" in f:
            return d
    raise RuntimeError("no test root under /kaggle/input")


def find_weight_file(fname):
    # direct dataset mounts first; NEVER recursive-glob the competitions DICOM tree.
    direct = [f"/kaggle/input/raptor-knee-arms/{fname}",
              f"/kaggle/input/raptor-knee-arms/1/{fname}",
              f"/kaggle/input/raptor-cnn336/{fname}"]
    for p in direct:
        if os.path.exists(p):
            return p
    for d in sorted(glob.glob("/kaggle/input/*/")):
        # skip the competition data wherever Kaggle mounts it (/kaggle/input/competitions/<slug> OR /kaggle/input/<slug>):
        # recursively globbing its DICOM tree hung a commit run for 20+ min (2026-09-26)
        if "competition" in d.lower() or "abnormality-detection" in d or any(
                os.path.exists(os.path.join(d, x)) for x in ("test_series", "train_images", "train.csv", "test.csv")):
            continue
        hits = glob.glob(os.path.join(d, "**", fname), recursive=True)
        if hits:
            return hits[0]
    raise RuntimeError(f"{fname} not found under /kaggle/input")


def main():
    import pandas as pd
    t0 = time.time()
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    ngpu = torch.cuda.device_count()
    print(f"device {dev} | gpus {ngpu} | torch {torch.__version__}", flush=True)

    ROOT = find_test_root()
    tsdir = ROOT + "/test_series"
    if not os.path.isdir(tsdir):
        tsdir = ROOT + "/test_images"
    print("test root:", ROOT, "| series dir:", tsdir, flush=True)

    test = pd.read_csv(ROOT + "/test.csv"); test["StudyInstanceUID"] = test["StudyInstanceUID"].astype(str)
    test_ids = test["StudyInstanceUID"].tolist()
    tser = pd.read_csv(ROOT + "/test_series.csv")
    tser["StudyInstanceUID"] = tser["StudyInstanceUID"].astype(str)
    tser["SeriesInstanceUID"] = tser["SeriesInstanceUID"].astype(str)
    SER = {k: v.to_dict("records") for k, v in tser.groupby("StudyInstanceUID")}
    print(f"test studies {len(test_ids)} | test series {len(tser)}", flush=True)

    sub_cols = ["StudyInstanceUID"] + LAB
    ssub = os.path.join(ROOT, "sample_submission.csv")
    if os.path.exists(ssub):
        sub_cols = list(pd.read_csv(ssub, nrows=1).columns)

    N = len(test_ids); A = len(ARMS)
    arm_probs = [np.full((N, len(LAB)), 0.5, np.float32) for _ in range(A)]
    pos = {sid: i for i, sid in enumerate(test_ids)}

    # ---- 2. GPU consumers: one thread per GPU, every arm loaded on every GPU (per-study batch of 1,
    #         so the math is identical to the sequential kernel; only the wall-clock changes) ----
    xla = None
    try:
        import torch_xla; import torch_xla.core.xla_model as xm
        dev = torch_xla.device(); xla = xm
        print("device: XLA/TPU", dev, flush=True)
    except Exception as e:
        print("XLA unavailable:", type(e).__name__, str(e)[:80], "| device", dev, flush=True)
    devs = [f"cuda:{g}" for g in range(ngpu)] if (xla is None and ngpu > 0) else [dev]
    models = {}
    for d in devs:
        models[d] = []
        for a, arm in enumerate(ARMS):
            model, res = load_model(find_weight_file(arm["file"]), arm["arch"], arm["res"], d)
            models[d].append((model, res))
        print(f"[gpu {d}] loaded {A} arm(s) | {time.time()-t0:.0f}s", flush=True)
    q = queue.Queue(maxsize=QUEUE_MAX); done_n = [0]; lock = threading.Lock()

    def consumer(d):
        while True:
            item = q.get()
            if item is None: q.task_done(); return
            sid, vol, mask = item; i = pos[sid]
            try:
                for a, (model, res) in enumerate(models[d]):
                    xw = eval_windows(vol, mask, k=K_EVAL, res=res, norm=NORM) if xla is not None else gpu_windows(vol, mask, K_EVAL, res, d)
                    assert xla is None, "Anatomical L/R TTA requires CUDA/fp16"
                    xm = gpu_windows(vol, mask, K_EVAL, res, d, norm=NORM, mirror=True)
                    arm_probs[a][i] = infer_pair_probs(model, xw, xm, d)
                    del xm
            except Exception as e:
                print(f"  [gpu {d}] study {i} {sid[:16]} FALLBACK ({type(e).__name__}: {e})", flush=True)
            with lock:
                done_n[0] += 1
                if done_n[0] % 200 == 0 or done_n[0] == N:
                    print(f"  [gpu] {done_n[0]}/{N} | {time.time()-t0:.0f}s", flush=True)
            q.task_done()

    threads = [threading.Thread(target=consumer, args=(d,), daemon=True) for d in devs]
    for th in threads: th.start()

    # ---- 3. prep every study ONCE in a process pool and stream it to the GPUs as it lands ----
    import multiprocessing as mp
    global _SER, _TSDIR, _READER
    _SER, _TSDIR = SER, tsdir
    nproc = max(4, min(96, (os.cpu_count() or 8) - 8))
    t_pre = time.time()
    with mp.get_context("fork").Pool(nproc, initializer=_pre_init) as pool:
        for i, (sid, vol, mask) in enumerate(pool.imap_unordered(_pre_one, test_ids, chunksize=2)):
            q.put((sid, vol, mask))
            if (i + 1) % 200 == 0 or i + 1 == N:
                print(f"  [pre] {i+1}/{N} | {time.time()-t0:.0f}s", flush=True)
    print(f"[pre] done: {N} studies in {time.time()-t_pre:.0f}s (procs {nproc} x fastread {FASTREAD_VARIANT} {FR_HDR}/{FR_PX} threads)", flush=True)
    for _ in devs: q.put(None)
    q.join()
    for th in threads: th.join()
    print(f"[gpu] done: {done_n[0]}/{N} studies on {len(devs)} device(s) | {time.time()-t0:.0f}s", flush=True)
    for d in devs:
        for m, _ in models[d]: del m
    models.clear(); gc.collect()

    # WEIGHTED rank-mean blend across the test set, per finding (the offline recipe).
    # Weights come from ARMS[*]["w"] and are normalised here, so dropping/adding an arm can
    # never silently change the scale. Falls back to equal weights if none are declared.
    _w = np.array([float(a.get("w", 1.0)) for a in ARMS], dtype=np.float64)
    _w = _w / _w.sum()
    print(f"[blend] weighted rank-mean w={dict(zip([a['file'] for a in ARMS], _w.round(4)))}", flush=True)
    ranks = np.tensordot(_w, np.stack([rankpct(np.clip(p, 0, 1)) for p in arm_probs]),
                         axes=(0, 0))                                          # (N,12) in [0,1]
    if not np.isfinite(ranks).all():
        ranks[~np.isfinite(ranks)] = 0.5

    sub = pd.DataFrame(ranks.astype(np.float32), columns=LAB)
    sub.insert(0, "StudyInstanceUID", test_ids)
    sub = sub[sub_cols]
    assert list(sub.columns) == sub_cols, "column order drift"
    assert sub["StudyInstanceUID"].tolist() == test_ids, "row identity drift"
    assert np.isfinite(sub[LAB].values).all()
    out = "/kaggle/working/submission.csv"
    sub.to_csv(out, index=False)
    print("wrote", out, "|", len(sub), "rows x", len(sub.columns), "cols", flush=True)
    print(sub.head().to_string(index=False), flush=True)
    print(f"DONE {time.time()-t0:.0f}s", flush=True)


if __name__ == "__main__":
    main()


In [ ]:
# ---- Part A2: nartaa's 0.945 efficiency checkpoint (224 crop), written to eff_sub.csv ----
open('/tmp/eff_run.py', 'w').write('# Authorized scoring run: verify the selected checkpoint before inference.\nRUN_INFERENCE = True\nif not RUN_INFERENCE:\n    raise SystemExit("Static notebook only. Attach the exact checkpoint and competition, enable a T4 GPU, and set RUN_INFERENCE = True to run inference.")\n\nimport hashlib\nfrom pathlib import Path\nimport torch\nif not torch.cuda.is_available():\n    raise RuntimeError("This scored pipeline requires CUDA; CPU inference is not enabled.")\n_weight_name = \'raptor_ft_alldata_t16_blendjev_oai_d96_c08_r224_swa.pt\'\n_candidates = [\n    Path(\'/kaggle/input/rsna-knee-publication-swa-weights-20261007\') / _weight_name,\n    Path(\'/kaggle/input/datasets/nartaa/rsna-knee-publication-swa-weights-20261007\') / _weight_name,\n    Path(\'/kaggle/input/rsna-knee-raptor-alldata-d96-blendjev-oai-c08r224\') / _weight_name,\n    Path(\'/kaggle/input/datasets/nartaa/rsna-knee-raptor-alldata-d96-blendjev-oai-c08r224\') / _weight_name,\n]\n_weights = next((p for p in _candidates if p.is_file()), None)\nif _weights is None:\n    raise FileNotFoundError("Attach the exact selected weights; no alternative model is substituted.")\nwith _weights.open(\'rb\') as _f:\n    assert hashlib.file_digest(_f, \'sha256\').hexdigest() == \'3394fdd7e668286956e85af2554bef0386df5cd79d1c1af223b095c828b815d6\', "Checkpoint hash differs"\n\n\n# %% [code]\n#!/usr/bin/env python3\n"""Knee MRI: twelve findings from a single model\n\nThis notebook takes a knee MRI study and scores twelve findings at once: ACL tear, MCL tear,\nmedial and lateral meniscus tears, osteoarthritis in the medial, lateral and patellofemoral\ncompartments, joint effusion, synovitis, a Baker\'s cyst, bone contusion and fracture. It scores\n0.924 on the public leaderboard using one model, with no ensembling and no test-time augmentation.\n\nThis is the inference half of the work. The model was trained separately and its weights are\nattached as a dataset, so this notebook only loads them and predicts:\nhttps://www.kaggle.com/datasets/dreaddevelopment/raptor-knee-widedense\n\nWhere the training labels came from\n\nWorth saying up front, because it shapes everything else. The competition gives you 4,407 studies\nbut structured labels for only 58 of them. Every other study arrives with a free-text radiology\nreport and nothing more, so there is very little to train against out of the box.\n\nThe labels behind these weights were made by reading those reports with a language model and\nturning each into twelve probabilities rather than twelve yes or no answers. A report that says a\ntear is suspected becomes a number near 0.8, not a 1, which is a fairer target than forcing every\nhedged sentence into a hard label. That yields 4,349 studies to train on. The 58 studies that came\nwith real labels were never trained on and are used to check the result honestly; the model reaches\n0.9167 macro-AUC on them.\n\nBuilding a fixed input from studies that are all shaped differently\n\nThe hard part of this competition is not the network, it is that no two studies look alike. A\nstudy holds several DICOM series shot in different planes, the number of series varies, and the\nnumber of slices in a series varies more. Anything that expects a fixed-size input has to be given\none.\n\nThe approach here is to fill five fixed slots per study, always in the same order, for a stack of\n64 images:\n\n  18 slices from a sagittal series, preferring a fluid-sensitive one\n  14 slices from a second sagittal series, preferring one that is not fluid-sensitive\n  12 slices from a coronal series, preferring a fluid-sensitive one\n   8 slices from a second coronal series\n  12 slices from an axial series\n\nPreferring a fluid-sensitive series for some slots and not for others is deliberate. Fluid-\nsensitive sequences show swelling, effusion and acute injury clearly, while the other sequences\nshow anatomy and cartilage better, and the twelve findings are split across both. If a study has\nno series for a slot, the slot is left as zeros and the model is told to skip it rather than being\nfed something misleading.\n\nWithin a series, slices are taken evenly across 6 to 94 percent of the stack rather than from the\nmiddle. The outer slices are where the collateral ligaments and the lateral meniscus sit, and\ncutting them was measurably costing accuracy on exactly those findings.\n\nEvery slice is cropped to a 140 mm box around the centre of the image using the pixel spacing from\nthe DICOM header, then resized to 336 pixels. Cropping by millimetres rather than by pixel count\nmatters: it means a knee occupies the same fraction of the frame whether the scan was acquired at\n0.3 or 0.5 mm per pixel, so the model is not asked to learn scale differences that carry no medical\ninformation.\n\nHow the model reads the stack\n\nThree neighbouring slices are stacked into the three channels of one image. The network then sees\na little of what lies above and below the slice in the middle, which is most of the benefit of a 3D\nmodel at the cost of a 2D one. Each of these three-slice windows is passed through a CoAtNet\nbackbone at 384 pixels.\n\nThe windows are combined with an attention layer that has separate weights for each of the twelve\nfindings. This is the part that matters most. A cruciate tear may be visible on two sagittal slices\nwhile osteoarthritis is spread across many coronal ones, and a single pooled score forces those two\nto share one notion of which slices are important. Giving each finding its own attention weights\nlets each one draw on the slices that actually show it.\n\nRunning it\n\nScoring uses 42 windows per study. Inference runs in half precision and automatically retries a\nstudy in full precision if it fails, so no study is ever dropped from the submission. The notebook\nneeds no internet: the backbone is loaded from the attached weights rather than downloaded.\n"""\nimport os, sys, glob, time, json, gc\nos.environ.setdefault("HF_HUB_OFFLINE", "1")\nos.environ.setdefault("TRANSFORMERS_OFFLINE", "1")\nos.environ.setdefault("HF_HUB_DISABLE_TELEMETRY", "1")\nimport numpy as np\nfrom concurrent.futures import ThreadPoolExecutor\nimport threading, queue\nimport torch, torch.nn as nn, torch.nn.functional as F\nimport timm\n# T4 (Turing) cuDNN v9 has fp16/fp32 conv engines but NOT bf16 for these shapes\n# ("GET was unable to find an engine..."); benchmark lets it pick a valid algo for\n# the fixed (1,24,3,res,res) input.\ntorch.backends.cudnn.benchmark = True\ntorch.backends.cuda.matmul.allow_tf32 = True\n\n# ---- fixed config (must match training exactly) -----------------------------\nIMG = 384   # native 384 corpus: the model saw 384 px crops directly, no 336->384 upsample\nCROP_MM = 140.0\n# 96 slices per study at native 384 px. Must match the corpus the weights were trained on\n# (cache/build_corpus96.py).\nSLOTS = [("Sagittal", 1, 26), ("Sagittal", 0, 22), ("Coronal", 1, 18),\n         ("Coronal", 0, 12), ("Axial", -1, 18)]\nMAXS = sum(s[2] for s in SLOTS)                     # 96\nK_EVAL = 94   # every window position of the 96-slice test volume (full coverage)\nHDR_THREADS = 16   # per prep process: header reads are fuse-latency-bound, not CPU-bound\nPX_THREADS = 8     # per prep process: pixel reads of the ~96 picked slices\nQUEUE_MAX = 48     # studies in flight between the prep pool and the GPU consumers (RAM bound: ~14 MB each)\nNORM = "imagenet"\nFASTREAD_VARIANT = \'study_lut\'   # kernel v3: research/fastread/readers.py, byte-identical to build_study\nFR_HDR, FR_PX = 32, 16\n_FASTREAD_B64 = "..."\ndef _load_fastread():\n    import base64 as _b, importlib.util as _iu\n    p = \'/kaggle/working/_fastread.py\' if os.path.isdir(\'/kaggle/working\') else os.path.join(os.path.dirname(os.path.abspath(__file__)), \'_fastread.py\')\n    if not os.path.exists(p):\n        open(p, \'wb\').write(_b.b64decode(_FASTREAD_B64))\n    spec = _iu.spec_from_file_location(\'_fastread\', p); m = _iu.module_from_spec(spec); spec.loader.exec_module(m)\n    return m\nLAB = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA",\n       "PF OA", "Effusion", "Synovitis", "Baker\'s", "Contusion", "Fracture"]\nCROP_FRAC = 0.8   # centre crop of the field before the resize (train_knee RAPTOR_CROP_FRAC)\ndef _crop(t):\n    H, W = t.shape[-2], t.shape[-1]; ch, cw = int(round(H * CROP_FRAC)), int(round(W * CROP_FRAC))\n    y0, x0 = (H - ch) // 2, (W - cw) // 2\n    return t[..., y0:y0 + ch, x0:x0 + cw]\n_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)\n_STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)\n\n# Three arms: (weights filename, fallback arch, fallback res). ck carries arch+res too.\n# Selected 2026-08-19 by greedy forward selection AND exhaustive subset search over a 7-arm\n# panel on the 45-study gold set (phase2/blend_panel.py); both agree on this exact set.\n# Singles: coatnet384 0.9025 | swinbase384 0.8825 | effv2l480 0.8716.\n# Blend {coatnet+swin+effv2l} = 0.9068 (2-arm {coatnet+swin} = 0.9059, coatnet alone 0.9025).\n# Dropped as redundant: cnn336 (0.8833, the former champion), cnbase384 (0.8754),\n# cnlarge384 (0.8752), maxvit384 (0.8438).\n#\n# SINGLE ARM: coatnet_rmlp_2_rw_384 retrained on the EXPANDED 4,349-study corpus.\n#\n# Why one arm and not the 3-arm blend: on the live leaderboard CoAtNet alone scored 0.914 while\n# every blend scored 0.914-0.915, so ensembling is worth ~+0.001 there -- the ~+0.010 it showed\n# on the old 45-study gold set was gold-set noise. One arm is also 1/3 the kernel runtime.\n#\n# Corpus expansion: the corpus previously held 3,200 of the 4,349 labelled studies and only 45\n# of the 58 gold studies. Rebuilt to 4,407 studies (+37.8% training data, 58-study gate).\n#\n# Measured on the 58-study gate (the incumbent re-scored on the SAME gate for a fair compare):\n#   incumbent CoAtNet (3,155-study corpus) 0.8923\n#   this model       (4,349-study corpus) 0.9054   (+0.0131, better in 92.7% of 2000 bootstraps)\n# Biggest gains land on the findings that were capping us: Lateral Meniscus +0.071,\n# Fracture +0.057, Lateral OA +0.048, Medial Meniscus +0.035, ACL +0.028.\nARMS = [   # round 2: native-res retrain with a centre crop of the 140 mm field\n    {"file": "raptor_ft_alldata_t16_blendjev_oai_d96_c08_r224_swa.pt", "arch": "coatnet_rmlp_2_rw_224.sw_in12k_ft_in1k", "res": 224, "w": 1.0},\n]\n\n\n# ============================================================================\n# Model -- verbatim from finetune_raptor.py\n# ============================================================================\ndef build_backbone(arch, pretrained=False):\n    # maxvit/maxxvit/coatnet are conv-attention hybrids: NO CLS token, NO interpolatable\n    # pos-embed -> avg pool. The "vit" substring in "coatnet"/"maxvit" must NOT route them\n    # down the ViT path (mirrors finetune_raptor.py exactly).\n    hybrid = arch.startswith(("maxvit", "maxxvit", "coatnet", "coat_", "convnext"))\n    is_vit = (not hybrid) and any(k in arch for k in ("vit", "deit", "dinov2", "eva", "beit"))\n    kw = dict(pretrained=pretrained, num_classes=0, in_chans=3)\n    if is_vit:\n        kw.update(global_pool="token", dynamic_img_size=True)\n    else:\n        kw.update(global_pool="avg")\n    return timm.create_model(arch, **kw)\n\n\nclass RaptorClassifier(nn.Module):\n    def __init__(self, backbone, F_dim=768, n=12, drop=0.2):\n        super().__init__()\n        self.backbone = backbone\n        self.norm = nn.LayerNorm(F_dim)\n        self.att = nn.Sequential(nn.Linear(F_dim, 256), nn.Tanh(), nn.Dropout(drop),\n                                 nn.Linear(256, n))\n        self.clsW = nn.Parameter(torch.zeros(n, F_dim))\n        self.clsb = nn.Parameter(torch.zeros(n))\n        nn.init.trunc_normal_(self.clsW, std=0.02)\n        self.n = n\n\n    def encode(self, x):\n        B, K = x.shape[:2]\n        f = self.backbone(x.flatten(0, 1))\n        return f.view(B, K, -1)\n\n    def head(self, feats):\n        h = self.norm(feats)\n        a = self.att(h)\n        a = torch.softmax(a, dim=1)\n        pooled = torch.einsum("bkn,bkf->bnf", a, h)\n        logits = (pooled * self.clsW).sum(-1) + self.clsb\n        return logits\n\n    def forward(self, x):\n        return self.head(self.encode(x))\n\n\ndef load_model(pt_path, arch_default, res_default, device, ngpu=1):\n    ck = torch.load(pt_path, map_location="cpu", weights_only=False)\n    arch = ck.get("arch", arch_default)\n    ck_res = int(ck.get("res", res_default))\n    bb = build_backbone(arch, pretrained=False)\n    model = RaptorClassifier(bb, F_dim=bb.num_features)\n    model.load_state_dict(ck["model"], strict=True)\n    model.eval().to(device)\n    # NOTE: DataParallel removed on purpose. On the full hidden test it drove a system-RAM OOM\n    # (per-forward module replication over many studies); a single T4 handles K_EVAL=24 windows\n    # fine. Arms are also run SEQUENTIALLY (see main) so peak RAM == one model, not two.\n    del ck\n    gc.collect()\n    return model, ck_res\n\n\n# ============================================================================\n# Eval windowing -- verbatim from finetune_raptor.py StudyWindows (train=False)\n# ============================================================================\ndef _eval_centers(mask, D, k):\n    valid = np.where(mask > 0)[0]\n    if len(valid) < 3:\n        valid = np.arange(min(3, D))\n    lo, hi = int(valid.min()), int(valid.max())\n    cs = [c for c in range(lo + 1, hi) if c - 1 >= lo and c + 1 <= hi]\n    if not cs:\n        cs = [max(1, min((lo + hi) // 2, D - 2))]\n    idx = np.linspace(0, len(cs) - 1, k).round().astype(int)\n    return [cs[i] for i in idx]\n\n\ndef eval_windows(vol, mask, k, res, norm=NORM):\n    D = vol.shape[0]\n    cs = _eval_centers(mask, D, k)\n    wins = np.empty((len(cs), 3, res, res), np.float32)\n    for j, c in enumerate(cs):\n        c = max(1, min(c, D - 2))\n        tri = np.stack([vol[c - 1], vol[c], vol[c + 1]], 0).astype(np.float32) / 255.0\n        t = _crop(torch.from_numpy(tri)) if CROP_FRAC < 1.0 else torch.from_numpy(tri)\n        if t.shape[-1] != res:\n            t = F.interpolate(t[None], size=(res, res), mode="bilinear",\n                              align_corners=False)[0]\n        wins[j] = t.numpy()\n    x = torch.from_numpy(wins)\n    if norm == "imagenet":\n        x = (x - _MEAN) / _STD\n    return x\n\n\n_WLUT = {}\ndef gpu_windows(vol, mask, k, res, dev, norm=NORM):\n    """Same windows as eval_windows, built on the GPU from the uint8 volume. Values come from a per-channel\n    256-entry table computed on the CPU with eval_windows\' own float32 ops, so they are bit-identical."""\n    D = vol.shape[0]\n    cs = np.array([max(1, min(c, D - 2)) for c in _eval_centers(mask, D, k)])\n    tri = torch.from_numpy(np.ascontiguousarray(vol[np.stack([cs - 1, cs, cs + 1], 1)])).to(dev)   # (k,3,IMG,IMG) uint8\n    key = (str(dev), norm)\n    if key not in _WLUT:\n        v = torch.from_numpy((np.arange(256, dtype=np.uint8).astype(np.float32) / 255.0))\n        t = torch.stack([v, v, v], 0).view(3, 256, 1)\n        if norm == \'imagenet\':\n            t = (t - _MEAN.view(3, 1, 1)) / _STD.view(3, 1, 1)\n        _WLUT[key] = t.view(3, 256).contiguous().to(dev)\n    L = _WLUT[key]\n    x = torch.stack([L[c][tri[:, c].long()] for c in range(3)], 1)                   # exact float32 values\n    if x.shape[-1] != res or CROP_FRAC < 1.0:\n        if norm == \'imagenet\':   # crop + resize BEFORE normalisation, as in eval_windows\n            L0 = _WLUT.setdefault((str(dev), \'none\'), torch.from_numpy(np.arange(256, dtype=np.uint8).astype(np.float32) / 255.0).to(dev))\n            x = L0[tri.long()]\n            if CROP_FRAC < 1.0: x = _crop(x)\n            x = F.interpolate(x, size=(res, res), mode=\'bilinear\', align_corners=False)\n            x = (x - _MEAN.to(dev)) / _STD.to(dev)\n        else:\n            x = F.interpolate(_crop(x) if CROP_FRAC < 1.0 else x, size=(res, res), mode=\'bilinear\', align_corners=False)\n    return x\n\n\n@torch.no_grad()\ndef infer_probs(model, xwins, device):\n    x = xwins.unsqueeze(0).to(device)\n    use_cuda = device != "cpu" and str(device).startswith("cuda")\n    if use_cuda:\n        # fp16 conv on T4 is fully cuDNN-supported (bf16 is NOT -> "no engine").\n        try:\n            with torch.autocast("cuda", dtype=torch.float16):\n                o = torch.sigmoid(model(x).float())\n            return o[0].cpu().numpy()\n        except RuntimeError:\n            # fp32 always has a Turing conv engine; slower but never drops a study.\n            torch.cuda.empty_cache()\n            o = torch.sigmoid(model(x).float())\n            return o[0].cpu().numpy()\n    o = torch.sigmoid(model(x).float())\n    return o[0].cpu().numpy()\n\n\ndef rankpct(x):                                   # per-column percentile rank in [0,1]\n    order = x.argsort(0).argsort(0).astype(np.float64)\n    return order / max(1, (x.shape[0] - 1))\n\n\n# ============================================================================\n# Preprocessing -- verbatim from kprep2/dino_preprocess.py, retargeted to TEST\n# ============================================================================\ndef _make_reader():\n    import pydicom, cv2\n    from pydicom.pixel_data_handlers.util import apply_modality_lut\n\n    def _hdr(f):\n        try:\n            h = pydicom.dcmread(f, stop_before_pixels=True)\n            iop = getattr(h, \'ImageOrientationPatient\', None)\n            ipp = getattr(h, \'ImagePositionPatient\', None)\n            if iop is not None and ipp is not None and len(iop) == 6:\n                r = np.array(iop[:3], float); c = np.array(iop[3:], float)\n                n = np.cross(r, c); pos = float(np.dot(np.array(ipp, float), n))\n            else:\n                pos = float(getattr(h, \'InstanceNumber\', 0) or 0)\n            ps = getattr(h, \'PixelSpacing\', None); ps = float(ps[0]) if ps is not None else 0.5\n            return (pos, f, ps, True)\n        except Exception:\n            return (0.0, f, 0.5, False)\n\n    def order_and_meta(sdir):\n        # Header reads are latency-bound on Kaggle\'s mounted competition data (~240k opens on the hidden\n        # set); a thread pool hides that latency. Records and sort are unchanged -> identical ordering.\n        fs = glob.glob(sdir + "/*.dcm")\n        with ThreadPoolExecutor(max_workers=HDR_THREADS) as ex:\n            out = list(ex.map(_hdr, fs))\n        recs = [(pos, f, ps) for pos, f, ps, _ in out]; ps_list = [ps for _, _, ps, ok in out if ok]\n        recs.sort(key=lambda x: x[0])\n        med_ps = float(np.median(ps_list)) if ps_list else 0.5\n        return [(f, ps) for _, f, ps in recs], med_ps\n\n    def read_px(f):\n        d = pydicom.dcmread(f)\n        a = apply_modality_lut(d.pixel_array, d).astype(np.float32)\n        if str(getattr(d, \'PhotometricInterpretation\', \'\')) == \'MONOCHROME1\':\n            a = a.max() - a\n        return a\n\n    def mm_crop_resize(a, ps):\n        h, w = a.shape; cpx = int(round(CROP_MM / max(ps, 1e-3)))\n        cpx = min(cpx, min(h, w)); y0 = (h - cpx) // 2; x0 = (w - cpx) // 2\n        a = a[y0:y0 + cpx, x0:x0 + cpx]\n        return cv2.resize(a, (IMG, IMG), interpolation=cv2.INTER_AREA)\n\n    return order_and_meta, read_px, mm_crop_resize\n\n\ndef _pick_series_for_slot(rows, plane, fluid, used):\n    cands = [r for r in rows if r[\'Anatomical_Plane\'] == plane and r[\'SeriesInstanceUID\'] not in used]\n    if fluid in (0, 1):\n        pref = [r for r in cands if int(r.get(\'Fluid_Sensitive\', 0) or 0) == fluid]\n        if pref:\n            return pref[0]\n    return cands[0] if cands else None\n\n\ndef build_study(sid, ser_records, tsdir, reader):\n    order_and_meta, read_px, mm_crop_resize = reader\n    rows = ser_records.get(sid, [])\n    vol = np.zeros((MAXS, IMG, IMG), np.uint8); idx = 0; used = set()\n    for plane, fluid, k in SLOTS:\n        r = _pick_series_for_slot(rows, plane, fluid, used)\n        if r is None:\n            idx += k; continue\n        used.add(r[\'SeriesInstanceUID\'])\n        files, med_ps = order_and_meta(f"{tsdir}/{sid}/{r[\'SeriesInstanceUID\']}")\n        if not files:\n            idx += k; continue\n        # wide span: the collateral ligaments and lateral meniscus live in the\n        # peripheral slices the old 0.15-0.85 crop threw away. Must match the corpus\n        # the weights were trained on (build_corpus80.py, SPAN).\n        n = len(files); lo, hi = int(n * 0.02), int(n * 0.98) - 1; hi = max(hi, lo)\n        picks = np.linspace(lo, hi, k).round().astype(int) if n > 1 else [0] * k\n        sel = [files[min(p, n - 1)] for p in picks]\n        def _px(fp_ps):\n            fp, ps = fp_ps\n            try: return read_px(fp), ps\n            except Exception: return None, med_ps\n        with ThreadPoolExecutor(max_workers=PX_THREADS) as ex:\n            got = list(ex.map(_px, sel))                    # order preserved\n        arrs = [a for a, _ in got]; pss = [ps for _, ps in got]\n        valid = [a for a in arrs if a is not None]\n        if valid:\n            allpx = np.concatenate([a.ravel() for a in valid])\n            loq, hiq = np.percentile(allpx, [2.0, 98.0])\n        else:\n            loq, hiq = 0.0, 1.0\n        for a, ps in zip(arrs, pss):\n            if idx >= MAXS: break\n            if a is None: idx += 1; continue\n            aw = np.clip((a - loq) / (hiq - loq + 1e-6), 0, 1)\n            aw = mm_crop_resize(aw, ps if ps > 0 else med_ps)\n            vol[idx] = (aw * 255).astype(np.uint8); idx += 1\n        if idx >= MAXS: break\n    mask = (vol.reshape(MAXS, -1).sum(1) > 0).astype(np.uint8)\n    return vol, mask\n\n\n\n# ---- parallel preprocessing helpers (module level so they pickle under fork) ----\n_SER = None; _TSDIR = None; _READER = None\ndef _pre_init():\n    global _READER\n    _READER = _load_fastread().make_builder(FASTREAD_VARIANT, IMG, SLOTS, hdr_threads=FR_HDR, px_threads=FR_PX)\ndef _pre_one(sid):\n    try:\n        vol, mask = _READER(sid, _SER, _TSDIR)\n    except Exception as e:\n        print(f"  [pre] {sid[:16]} FAILED ({type(e).__name__}: {e})", flush=True)\n        vol = np.zeros((MAXS, IMG, IMG), np.uint8); mask = np.zeros(MAXS, np.uint8)\n    return sid, vol, mask\n\n# ============================================================================\n# Test-root discovery + weights + main\n# ============================================================================\ndef find_test_root():\n    cands = ["/kaggle/input/competitions/rsna-knee-abnormality-detection",\n             "/kaggle/input/rsna-knee-abnormality-detection"]\n    for b in cands:\n        if os.path.exists(b + "/test.csv"):\n            return b\n    for d, _, f in os.walk("/kaggle/input"):\n        if "test.csv" in f and (os.path.isdir(d + "/test_series") or os.path.isdir(d + "/test_images")):\n            return d\n    for d, _, f in os.walk("/kaggle/input"):\n        if "test.csv" in f:\n            return d\n    raise RuntimeError("no test root under /kaggle/input")\n\n\ndef find_weight_file(fname):\n    # direct dataset mounts first; NEVER recursive-glob the competitions DICOM tree.\n    direct = [f"/kaggle/input/raptor-knee-arms/{fname}",\n              f"/kaggle/input/raptor-knee-arms/1/{fname}",\n              f"/kaggle/input/raptor-cnn336/{fname}"]\n    for p in direct:\n        if os.path.exists(p):\n            return p\n    for d in sorted(glob.glob("/kaggle/input/*/")):\n        # skip the competition data wherever Kaggle mounts it (/kaggle/input/competitions/<slug> OR /kaggle/input/<slug>):\n        # recursively globbing its DICOM tree hung a commit run for 20+ min (2026-09-26)\n        if "competition" in d.lower() or "abnormality-detection" in d or any(\n                os.path.exists(os.path.join(d, x)) for x in ("test_series", "train_images", "train.csv", "test.csv")):\n            continue\n        hits = glob.glob(os.path.join(d, "**", fname), recursive=True)\n        if hits:\n            return hits[0]\n    raise RuntimeError(f"{fname} not found under /kaggle/input")\n\n\ndef main():\n    import pandas as pd\n    t0 = time.time()\n    dev = "cuda" if torch.cuda.is_available() else "cpu"\n    ngpu = torch.cuda.device_count()\n    print(f"device {dev} | gpus {ngpu} | torch {torch.__version__}", flush=True)\n\n    ROOT = find_test_root()\n    tsdir = ROOT + "/test_series"\n    if not os.path.isdir(tsdir):\n        tsdir = ROOT + "/test_images"\n    print("test root:", ROOT, "| series dir:", tsdir, flush=True)\n\n    test = pd.read_csv(ROOT + "/test.csv"); test["StudyInstanceUID"] = test["StudyInstanceUID"].astype(str)\n    test_ids = test["StudyInstanceUID"].tolist()\n    tser = pd.read_csv(ROOT + "/test_series.csv")\n    tser["StudyInstanceUID"] = tser["StudyInstanceUID"].astype(str)\n    tser["SeriesInstanceUID"] = tser["SeriesInstanceUID"].astype(str)\n    SER = {k: v.to_dict("records") for k, v in tser.groupby("StudyInstanceUID")}\n    print(f"test studies {len(test_ids)} | test series {len(tser)}", flush=True)\n\n    sub_cols = ["StudyInstanceUID"] + LAB\n    ssub = os.path.join(ROOT, "sample_submission.csv")\n    if os.path.exists(ssub):\n        sub_cols = list(pd.read_csv(ssub, nrows=1).columns)\n\n    N = len(test_ids); A = len(ARMS)\n    arm_probs = [np.full((N, len(LAB)), 0.5, np.float32) for _ in range(A)]\n    pos = {sid: i for i, sid in enumerate(test_ids)}\n\n    # ---- 2. GPU consumers: one thread per GPU, every arm loaded on every GPU (per-study batch of 1,\n    #         so the math is identical to the sequential kernel; only the wall-clock changes) ----\n    xla = None\n    try:\n        import torch_xla; import torch_xla.core.xla_model as xm\n        dev = torch_xla.device(); xla = xm\n        print("device: XLA/TPU", dev, flush=True)\n    except Exception as e:\n        print("XLA unavailable:", type(e).__name__, str(e)[:80], "| device", dev, flush=True)\n    devs = [f"cuda:{g}" for g in range(ngpu)] if (xla is None and ngpu > 0) else [dev]\n    models = {}\n    for d in devs:\n        models[d] = []\n        for a, arm in enumerate(ARMS):\n            model, res = load_model(find_weight_file(arm["file"]), arm["arch"], arm["res"], d)\n            models[d].append((model, res))\n        print(f"[gpu {d}] loaded {A} arm(s) | {time.time()-t0:.0f}s", flush=True)\n    q = queue.Queue(maxsize=QUEUE_MAX); done_n = [0]; lock = threading.Lock()\n\n    def consumer(d):\n        while True:\n            item = q.get()\n            if item is None: q.task_done(); return\n            sid, vol, mask = item; i = pos[sid]\n            try:\n                for a, (model, res) in enumerate(models[d]):\n                    xw = eval_windows(vol, mask, k=K_EVAL, res=res, norm=NORM) if xla is not None else gpu_windows(vol, mask, K_EVAL, res, d)\n                    if xla is not None:\n                        with torch.no_grad():\n                            o = torch.sigmoid(model(xw.unsqueeze(0).to(d)).float()); xla.mark_step()\n                            arm_probs[a][i] = o[0].cpu().numpy()\n                    else:\n                        arm_probs[a][i] = infer_probs(model, xw, d)\n            except Exception as e:\n                print(f"  [gpu {d}] study {i} {sid[:16]} FALLBACK ({type(e).__name__}: {e})", flush=True)\n            with lock:\n                done_n[0] += 1\n                if done_n[0] % 200 == 0 or done_n[0] == N:\n                    print(f"  [gpu] {done_n[0]}/{N} | {time.time()-t0:.0f}s", flush=True)\n            q.task_done()\n\n    threads = [threading.Thread(target=consumer, args=(d,), daemon=True) for d in devs]\n    for th in threads: th.start()\n\n    # ---- 3. prep every study ONCE in a process pool and stream it to the GPUs as it lands ----\n    import multiprocessing as mp\n    global _SER, _TSDIR, _READER\n    _SER, _TSDIR = SER, tsdir\n    nproc = max(4, min(96, (os.cpu_count() or 8) - 8))\n    t_pre = time.time()\n    with mp.get_context("fork").Pool(nproc, initializer=_pre_init) as pool:\n        for i, (sid, vol, mask) in enumerate(pool.imap_unordered(_pre_one, test_ids, chunksize=2)):\n            q.put((sid, vol, mask))\n            if (i + 1) % 200 == 0 or i + 1 == N:\n                print(f"  [pre] {i+1}/{N} | {time.time()-t0:.0f}s", flush=True)\n    print(f"[pre] done: {N} studies in {time.time()-t_pre:.0f}s (procs {nproc} x fastread {FASTREAD_VARIANT} {FR_HDR}/{FR_PX} threads)", flush=True)\n    for _ in devs: q.put(None)\n    q.join()\n    for th in threads: th.join()\n    print(f"[gpu] done: {done_n[0]}/{N} studies on {len(devs)} device(s) | {time.time()-t0:.0f}s", flush=True)\n    for d in devs:\n        for m, _ in models[d]: del m\n    models.clear(); gc.collect()\n\n    # WEIGHTED rank-mean blend across the test set, per finding (the offline recipe).\n    # Weights come from ARMS[*]["w"] and are normalised here, so dropping/adding an arm can\n    # never silently change the scale. Falls back to equal weights if none are declared.\n    _w = np.array([float(a.get("w", 1.0)) for a in ARMS], dtype=np.float64)\n    _w = _w / _w.sum()\n    print(f"[blend] weighted rank-mean w={dict(zip([a[\'file\'] for a in ARMS], _w.round(4)))}", flush=True)\n    ranks = np.tensordot(_w, np.stack([rankpct(np.clip(p, 0, 1)) for p in arm_probs]),\n                         axes=(0, 0))                                          # (N,12) in [0,1]\n    if not np.isfinite(ranks).all():\n        ranks[~np.isfinite(ranks)] = 0.5\n\n    sub = pd.DataFrame(ranks.astype(np.float32), columns=LAB)\n    sub.insert(0, "StudyInstanceUID", test_ids)\n    sub = sub[sub_cols]\n    assert list(sub.columns) == sub_cols, "column order drift"\n    assert sub["StudyInstanceUID"].tolist() == test_ids, "row identity drift"\n    assert np.isfinite(sub[LAB].values).all()\n    out = "/kaggle/working/eff_sub.csv"\n    sub.to_csv(out, index=False)\n    print("wrote", out, "|", len(sub), "rows x", len(sub.columns), "cols", flush=True)\n    print(sub.head().to_string(index=False), flush=True)\n    print(f"DONE {time.time()-t0:.0f}s", flush=True)\n\n\nif __name__ == "__main__":\n    main()\n')


In [ ]:
# Part A2 run + rank blend into Part A's submission.csv (Part B then reads the blended file).
import gc, os, shutil, subprocess, sys, time
import numpy as np, pandas as pd
W_EFF = 0.25
try:
    import torch; gc.collect(); torch.cuda.empty_cache()
except Exception as _e:
    print('cuda cleanup skipped', _e)
shutil.copy('/kaggle/working/submission.csv', '/kaggle/working/_a_submission.csv')
_t0 = time.time()
try:
    _rc = subprocess.run([sys.executable, '-u', '/tmp/eff_run.py'], timeout=3 * 3600).returncode
    assert _rc == 0, 'eff leg exit code %d' % _rc
    _a = pd.read_csv('/kaggle/working/_a_submission.csv', dtype={'StudyInstanceUID': str})
    _e = pd.read_csv('/kaggle/working/eff_sub.csv', dtype={'StudyInstanceUID': str}).set_index('StudyInstanceUID').loc[_a.StudyInstanceUID].reset_index()
    _lab = [c for c in _a.columns if c != 'StudyInstanceUID']
    assert _e[_lab].notna().all().all() and np.isfinite(_e[_lab].values).all(), 'eff leg NaN'
    _r = lambda d: d[_lab].rank(method='average', pct=True)
    _out = _a.copy()
    _out[_lab] = ((1 - W_EFF) * _r(_a) + W_EFF * _r(_e)).rank(method='average', pct=True)
    _out.to_csv('/kaggle/working/submission.csv', index=False)
    print('Part A2: eff leg blended at', W_EFF, 'in %.0fs' % (time.time() - _t0),
          '| mean rank corr acc-eff', round(float(_r(_a).corrwith(_r(_e)).mean()), 4) if len(_a) > 2 else 'n<3', flush=True)
except Exception as _x:
    shutil.copy('/kaggle/working/_a_submission.csv', '/kaggle/working/submission.csv')
    print('Part A2 FAILED, Part A submission kept:', repr(_x), flush=True)


---
# Part B · our reader

The next cells write the reader's source to `/kaggle/working/own_src/` and the last cell runs its three checkpoints and blends them in. If it fails, Part A's submission is kept.

In [ ]:
import os
os.makedirs('/kaggle/working/own_src', exist_ok=True)

In [ ]:
%%writefile /kaggle/working/own_src/preprocess.py
"""Turn each DICOM series into a canonical uint8 volume [n_slices, SIZE, SIZE].

Canonical means: fixed physical pixel size, centre crop to a fixed field of view,
standard in-plane orientation, standard slice order and a per-series intensity window.
The same `load_series` is used for the training cache and for inference.

Usage: python src/preprocess.py [train|test] [n_series_limit]
"""
import os
import sys
from multiprocessing import Pool

import cv2
import numpy as np
import pandas as pd
import pydicom

DATA = "/mnt/ssd/rsna_knee/data"
WORK = "/mnt/ssd/rsna_knee/work"

MM_PER_PX = 0.4
SIZE = 384  # 153.6 mm field of view
MAX_SLICES = 64

# plane -> (normal axis, slice-order sign, (col axis, sign), (row axis, sign)) in LPS coordinates
# Sagittal: slices right->left, image columns anterior->posterior, rows superior->inferior
# Coronal:  slices anterior->posterior, columns patient-right->left, rows superior->inferior
# Axial:    slices superior->inferior, columns patient-right->left, rows anterior->posterior
CANON = {
    0: (+1, (1, +1), (2, -1)),
    1: (+1, (0, +1), (2, -1)),
    2: (-1, (0, +1), (1, +1)),
}
PLANES = ["Sagittal", "Coronal", "Axial"]


def _read(path):
    ds = pydicom.dcmread(path)
    arr = ds.pixel_array.astype(np.float32)
    if ds.get("PhotometricInterpretation", "MONOCHROME2") == "MONOCHROME1":
        arr = arr.max() - arr
    slope, inter = ds.get("RescaleSlope"), ds.get("RescaleIntercept")
    if slope is not None and inter is not None:
        arr = arr * float(slope) + float(inter)
    iop = [float(v) for v in ds.ImageOrientationPatient] if "ImageOrientationPatient" in ds else None
    ipp = [float(v) for v in ds.ImagePositionPatient] if "ImagePositionPatient" in ds else None
    ps = [float(v) for v in ds.PixelSpacing] if "PixelSpacing" in ds else None
    inst = float(ds.get("InstanceNumber") or 0)
    frames = list(arr) if arr.ndim == 3 else [arr]
    return [(f, iop, ipp, ps, inst + i * 1e-3) for i, f in enumerate(frames)]


def _canon_slice(img, iop, ps, plane):
    """Reorient one slice to the canonical in-plane orientation; returns (img, (row_mm, col_mm))."""
    ps = ps or [MM_PER_PX * img.shape[0] / SIZE] * 2  # no spacing: assume the image spans the crop
    if iop is None:
        return img, ps
    r, c = np.array(iop[:3]), np.array(iop[3:])
    _, (cax, csgn), (rax, rsgn) = CANON[plane]
    if abs(r[cax]) < abs(r[rax]):  # rows and columns are swapped relative to canonical
        img, r, c, ps = img.T, c, r, ps[::-1]
    if r[cax] * csgn < 0:
        img = img[:, ::-1]
    if c[rax] * rsgn < 0:
        img = img[::-1]
    return img, ps


def _resample(img, ps):
    """Centre crop to SIZE*MM_PER_PX mm, resample to MM_PER_PX, zero-pad to SIZE x SIZE."""
    out = np.zeros((SIZE, SIZE), np.float32)
    h, w = img.shape
    ch = min(h, int(round(SIZE * MM_PER_PX / ps[0])))
    cw = min(w, int(round(SIZE * MM_PER_PX / ps[1])))
    y0, x0 = (h - ch) // 2, (w - cw) // 2
    crop = np.ascontiguousarray(img[y0:y0 + ch, x0:x0 + cw])
    nh = max(1, min(SIZE, int(round(ch * ps[0] / MM_PER_PX))))
    nw = max(1, min(SIZE, int(round(cw * ps[1] / MM_PER_PX))))
    interp = cv2.INTER_AREA if nh < ch else cv2.INTER_LINEAR
    res = cv2.resize(crop, (nw, nh), interpolation=interp)
    oy, ox = (SIZE - nh) // 2, (SIZE - nw) // 2
    out[oy:oy + nh, ox:ox + nw] = res
    return out


def load_series(sdir, plane_name=None):
    """Returns (uint8 volume [n, SIZE, SIZE], info dict). Unreadable slices are skipped."""
    slices, bad = [], 0
    for fn in os.listdir(sdir):
        try:
            slices.extend(_read(os.path.join(sdir, fn)))
        except Exception:
            bad += 1
    info = {"n_raw": len(slices), "n_bad": bad}
    if not slices:
        return np.zeros((1, SIZE, SIZE), np.uint8), info

    iop = next((s[1] for s in slices if s[1] is not None), None)
    if iop is not None:
        normal = np.cross(iop[:3], iop[3:])
        plane = int(np.abs(normal).argmax())
    else:
        normal, plane = None, PLANES.index(plane_name) if plane_name in PLANES else 0
    info["plane"] = PLANES[plane]

    if normal is not None and all(s[2] is not None for s in slices):
        sgn = CANON[plane][0] * np.sign(normal[plane])
        slices.sort(key=lambda s: float(np.dot(s[2], normal)) * sgn)
        pos = [float(np.dot(s[2], normal)) for s in slices]
        info["slice_gap"] = float(np.median(np.abs(np.diff(pos)))) if len(pos) > 1 else 0.0
    else:
        slices.sort(key=lambda s: s[4])
    if len(slices) > MAX_SLICES:
        keep = np.linspace(0, len(slices) - 1, MAX_SLICES).round().astype(int)
        slices = [slices[i] for i in keep]

    vol = np.stack([_resample(*_canon_slice(s[0], s[1], s[3], plane)) for s in slices])
    lo, hi = np.percentile(vol[:, ::4, ::4], [1, 99.5])
    vol = np.clip((vol - lo) / max(hi - lo, 1e-6), 0, 1)
    info["n"] = len(vol)
    return (vol * 255).round().astype(np.uint8), info


def _job(args):
    split, study, series, plane, out_dir = args
    out = f"{out_dir}/{series}.npy"
    info = {"study": study, "series": series}
    if os.path.exists(out):
        return None
    try:
        vol, extra = load_series(f"{DATA}/{split}_series/{study}/{series}", plane)
        np.save(out + ".tmp.npy", vol)
        os.replace(out + ".tmp.npy", out)
        info.update(extra)
    except Exception as e:
        info["error"] = f"{type(e).__name__}: {e}"[:200]
    return info


def main():
    split = sys.argv[1] if len(sys.argv) > 1 else "train"
    limit = int(sys.argv[2]) if len(sys.argv) > 2 else None
    out_dir = f"{WORK}/cache{SIZE}_{split}"
    os.makedirs(out_dir, exist_ok=True)
    se = pd.read_csv(f"{DATA}/{split}_series.csv")
    if limit:
        se = se.sample(limit, random_state=0)
    tasks = [(split, a, b, p, out_dir) for a, b, p in zip(se.StudyInstanceUID, se.SeriesInstanceUID, se.Anatomical_Plane)]
    rows = []
    with Pool(18) as p:
        for i, r in enumerate(p.imap_unordered(_job, tasks, chunksize=4)):
            if r:
                rows.append(r)
            if i % 1000 == 0:
                print(f"{i}/{len(tasks)}", flush=True)
    df = pd.DataFrame(rows)
    meta = f"{WORK}/cache{SIZE}_{split}_meta.csv"
    if os.path.exists(meta) and len(df):
        df = pd.concat([pd.read_csv(meta), df]).drop_duplicates("series", keep="last")
    if len(df):
        df.to_csv(meta, index=False)
        print("done", len(df), "errors:", int(df["error"].notna().sum()) if "error" in df else 0)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/own_src/knee.py
"""Shared dataset and model code for training and inference."""
import math

import numpy as np
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F

LABELS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA",
          "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]
PLANES = ["Sagittal", "Coronal", "Axial"]
# slot = plane index + 3 * (1 - fat_suppressed): fat-suppressed planes first
N_SLOTS = 6
MEAN = (0.485, 0.456, 0.406)
STD = (0.229, 0.224, 0.225)


def slot_of(plane, fat_sup):
    return PLANES.index(plane) + 3 * (1 - int(fat_sup))


def build_study_table(series_df, n_slices):
    """series_df: StudyInstanceUID, SeriesInstanceUID, Fat_Suppression, Anatomical_Plane.
    n_slices: dict series -> cached slice count. Returns {study: [[series ids of slot 0], ...]}
    with each slot's candidates sorted by slice count (largest first)."""
    table = {}
    for st, sr, fs, pl in zip(series_df.StudyInstanceUID, series_df.SeriesInstanceUID,
                              series_df.Fat_Suppression, series_df.Anatomical_Plane):
        if n_slices.get(sr, 0) < 3:
            continue
        table.setdefault(st, [[] for _ in range(N_SLOTS)])[slot_of(pl, fs)].append(sr)
    for slots in table.values():
        for cands in slots:
            cands.sort(key=lambda s: -n_slices[s])
        # Intelligent anatomical plane fallback: if fat-sat is missing, use non-fat-sat (and vice-versa)
        for p in range(3):
            fs_s, nonfs_s = p, p + 3
            if not slots[fs_s] and slots[nonfs_s]:
                slots[fs_s] = list(slots[nonfs_s])
            elif not slots[nonfs_s] and slots[fs_s]:
                slots[nonfs_s] = list(slots[fs_s])
    return table


def window_centres(n, k, train, lo=0.04, hi=0.96):
    """k slice indices (centres of 3-slice windows) spread over the series."""
    a, b = max(1.0, lo * (n - 1)), min(n - 2.0, hi * (n - 1))
    if b < a:
        a = b = (n - 1) / 2
    if train:
        edges = np.linspace(a, b, k + 1)
        c = edges[:-1] + np.random.rand(k) * (edges[1:] - edges[:-1])
    else:
        c = np.linspace(a, b, k)
    return np.clip(np.round(c).astype(int), 1, max(1, n - 2))


class StudyDataset(torch.utils.data.Dataset):
    """One item = one study: uint8 windows [N_SLOTS, k, 3, H, W], slot mask, slice positions, target."""

    def __init__(self, studies, table, cache_dir, targets=None, k=8, train=False, loader=None):
        self.studies, self.table, self.cache_dir = list(studies), table, cache_dir
        self.targets, self.k, self.train = targets, k, train
        self.loader = loader or (lambda s: np.load(f"{self.cache_dir}/{s}.npy", mmap_mode="r"))

    def __len__(self):
        return len(self.studies)

    def __getitem__(self, i):
        st = self.studies[i]
        slots = self.table.get(st, [[] for _ in range(N_SLOTS)])
        x, mask, pos = None, np.zeros(N_SLOTS, bool), np.zeros((N_SLOTS, self.k), np.float32)
        for s, cands in enumerate(slots):
            if not cands:
                continue
            sr = cands[np.random.randint(len(cands))] if self.train else cands[0]
            vol = self.loader(sr)
            n = len(vol)
            c = window_centres(n, self.k, self.train)
            win = np.stack([vol[np.clip([j - 1, j, j + 1], 0, n - 1)] for j in c])  # [k, 3, H, W]
            if x is None:
                x = np.zeros((N_SLOTS, self.k) + win.shape[1:], np.uint8)
            x[s], mask[s], pos[s] = win, True, c / max(n - 1, 1)
        if x is None:
            x = np.zeros((N_SLOTS, self.k, 3, 384, 384), np.uint8)
        y = self.targets[i] if self.targets is not None else np.zeros(len(LABELS), np.float32)
        return torch.from_numpy(x), torch.from_numpy(mask), torch.from_numpy(pos), torch.from_numpy(np.asarray(y, np.float32))


def make_views(x, res, train, group, fov=0.92):
    """x: uint8 [N, 3, H, W] with N = n_series * group. Crops a centred `fov` fraction of the cached
    field of view and resizes to res; in training adds rotation/scale/shift and gain/gamma jitter
    that is shared within a series."""
    n = x.shape[0]
    ns = n // group
    dev = x.device
    x = x.float().div_(255.0)
    if train:
        ang = (torch.rand(ns, device=dev) - 0.5) * 2 * math.radians(12)
        sc = fov * (1 + (torch.rand(ns, device=dev) - 0.5) * 0.24)
        tx = (torch.rand(ns, device=dev) - 0.5) * 0.12
        ty = (torch.rand(ns, device=dev) - 0.5) * 0.12
        gamma = torch.exp((torch.rand(ns, device=dev) - 0.5) * 0.5)
        gain = 1 + (torch.rand(ns, device=dev) - 0.5) * 0.2
    else:
        ang = torch.zeros(ns, device=dev)
        sc = torch.full((ns,), fov, device=dev)
        tx = ty = torch.zeros(ns, device=dev)
        gamma = gain = None
    cos, sin = torch.cos(ang) * sc, torch.sin(ang) * sc
    theta = torch.stack([torch.stack([cos, -sin, tx], 1), torch.stack([sin, cos, ty], 1)], 1)
    theta = theta.repeat_interleave(group, 0)
    grid = F.affine_grid(theta, (n, 3, res, res), align_corners=False)
    if x.shape[-1] > 1.5 * res:  # avoid aliasing when shrinking a lot
        x = F.avg_pool2d(x, 2) if x.shape[-1] >= 2 * res else F.interpolate(x, scale_factor=0.75, mode="bilinear", antialias=True)
    x = F.grid_sample(x, grid, mode="bilinear", padding_mode="zeros", align_corners=False)
    if train:
        g = gamma.repeat_interleave(group).view(n, 1, 1, 1)
        x = x.clamp_min(1e-4).pow(g) * gain.repeat_interleave(group).view(n, 1, 1, 1)
    mean = torch.tensor(MEAN, device=dev).view(1, 3, 1, 1)
    std = torch.tensor(STD, device=dev).view(1, 3, 1, 1)
    return (x - mean) / std


class KneeNet(nn.Module):
    """2D backbone on 3-slice windows -> tokens tagged with series slot and slice position ->
    small transformer over all windows of the study -> one attention pooling per finding."""

    def __init__(self, backbone="convnext_tiny", pretrained=True, dim=384, depth=2, drop=0.2, drop_path=0.1,
                 img_size=None):
        super().__init__()
        kw = {"drop_path_rate": drop_path} if drop_path else {}
        if img_size and backbone.startswith(("vit_", "eva", "beit")):  # fixed-grid transformers need the input size
            kw["img_size"] = img_size
        self.enc = timm.create_model(backbone, pretrained=pretrained, num_classes=0, **kw)
        self.proj = nn.Sequential(nn.Dropout(drop), nn.Linear(self.enc.num_features, dim), nn.LayerNorm(dim))
        self.slot_emb = nn.Embedding(N_SLOTS, dim)
        self.pos_emb = nn.Sequential(nn.Linear(1, dim), nn.GELU(), nn.Linear(dim, dim))
        layer = nn.TransformerEncoderLayer(dim, 8, dim * 2, dropout=0.1, batch_first=True, norm_first=True)
        self.tf = nn.TransformerEncoder(layer, depth, enable_nested_tensor=False) if depth else None
        self.att = nn.Sequential(nn.Linear(dim, 256), nn.Tanh(), nn.Dropout(drop), nn.Linear(256, len(LABELS)))
        self.cls_w = nn.Parameter(torch.randn(len(LABELS), dim) * 0.02)
        self.cls_b = nn.Parameter(torch.zeros(len(LABELS)))

    def head(self, feat, mask, pos):
        """feat [B, S, K, D], mask [B, S] bool, pos [B, S, K] in 0..1 -> logits [B, 12]."""
        b, s, k, _ = feat.shape
        slot = torch.arange(s, device=feat.device).view(1, s, 1).expand(b, s, k)
        h = self.proj(feat) + self.slot_emb(slot) + self.pos_emb(pos.unsqueeze(-1).to(feat.dtype))
        h = h.view(b, s * k, -1)
        pad = ~mask.unsqueeze(-1).expand(b, s, k).reshape(b, s * k)
        pad = pad & ~pad.all(1, keepdim=True)  # a study with no series at all attends to everything
        if self.tf is not None:
            h = self.tf(h, src_key_padding_mask=pad)
        a = self.att(h).float().masked_fill(pad.unsqueeze(-1), -1e4).softmax(1)  # [B, T, C]
        pooled = torch.einsum("btc,btd->bcd", a.to(h.dtype), h)
        return (pooled * self.cls_w).sum(-1) + self.cls_b

    def forward(self, x, mask, pos, res=256, train=False):
        """x uint8 [B, S, K, 3, H, W]."""
        b, s, k = x.shape[:3]
        keep = mask.view(-1)  # run the backbone only on series that exist
        xv = x.view(b * s, k, *x.shape[3:])[keep].flatten(0, 1)
        f = self.enc(make_views(xv, res, train, k).contiguous(memory_format=torch.channels_last))
        feat = f.new_zeros(b * s, k, f.shape[-1])
        feat[keep] = f.view(-1, k, f.shape[-1])
        return self.head(feat.view(b, s, k, -1), mask, pos)

In [ ]:
%%writefile /kaggle/working/own_src/infer.py
"""Predict the 12 findings for every study straight from DICOM folders.

Used by the Kaggle submission notebook and locally to check that predictions from raw DICOMs
match those from the training cache.

Usage: python src/infer.py <data_dir> <split> <out_csv> <ckpt> [<ckpt> ...]
  data_dir contains <split>.csv, <split>_series.csv and <split>_series/
"""
import os
import sys

import numpy as np
import pandas as pd
import torch

from knee import LABELS, N_SLOTS, KneeNet, build_study_table, window_centres
from preprocess import MAX_SLICES, SIZE, load_series


class DicomStudyDataset(torch.utils.data.Dataset):
    """Loads every selected series of a study once; windows are cut per model setting later."""

    def __init__(self, studies, table, root, k):
        self.studies, self.table, self.root, self.k = list(studies), table, root, k

    def __len__(self):
        return len(self.studies)

    def __getitem__(self, i):
        st = self.studies[i]
        slots = self.table.get(st, [[] for _ in range(N_SLOTS)])
        x = np.zeros((N_SLOTS, self.k, 3, SIZE, SIZE), np.uint8)
        mask, pos = np.zeros(N_SLOTS, bool), np.zeros((N_SLOTS, self.k), np.float32)
        for s, cands in enumerate(slots):
            if not cands:
                continue
            try:
                vol, _ = load_series(f"{self.root}/{st}/{cands[0]}")
            except Exception:
                continue
            n = len(vol)
            if n < 3:
                continue
            c = window_centres(n, self.k, False)
            x[s] = np.stack([vol[np.clip([j - 1, j, j + 1], 0, n - 1)] for j in c])
            mask[s], pos[s] = True, c / max(n - 1, 1)
        return torch.from_numpy(x), torch.from_numpy(mask), torch.from_numpy(pos)


def load_models(ckpts, device="cuda"):
    models = []
    for path in ckpts:
        ck = torch.load(path, map_location="cpu", weights_only=False)
        # the pretrained tag after the dot is irrelevant here and unknown to older timm versions
        m = KneeNet(ck["args"]["backbone"].split(".")[0], pretrained=False, img_size=ck["args"]["res"])
        m.load_state_dict(ck["model"])
        models.append((m.to(device).to(memory_format=torch.channels_last).eval(), ck["args"]["res"]))
    return models


@torch.no_grad()
def run(data_dir, split, ckpts, k=16, studies=None, workers=4, bs=4):
    se = pd.read_csv(f"{data_dir}/{split}_series.csv")
    root = f"{data_dir}/{split}_series"
    if studies is None:
        studies = pd.read_csv(f"{data_dir}/{split}.csv").StudyInstanceUID.tolist()
    se = se[se.StudyInstanceUID.isin(set(studies))]
    n_files = {}
    for st, sr in zip(se.StudyInstanceUID, se.SeriesInstanceUID):
        d = f"{root}/{st}/{sr}"
        n_files[sr] = min(len(os.listdir(d)), MAX_SLICES) if os.path.isdir(d) else 0
    table = build_study_table(se, n_files)
    dl = torch.utils.data.DataLoader(DicomStudyDataset(studies, table, root, k), batch_size=bs,
                                     num_workers=workers, pin_memory=True)
    models = load_models(ckpts)
    out = []
    for x, mask, pos in dl:
        x, mask, pos = x.cuda(non_blocking=True), mask.cuda(), pos.cuda()
        with torch.autocast("cuda", dtype=torch.float16):
            p = torch.stack([m(x, mask, pos, res=res).float().sigmoid() for m, res in models]).mean(0)
        out.append(p.cpu())
    return pd.DataFrame(torch.cat(out).numpy(), index=pd.Index(studies, name="StudyInstanceUID"), columns=LABELS)


if __name__ == "__main__":
    data_dir, split, out_csv, ckpts = sys.argv[1], sys.argv[2], sys.argv[3], sys.argv[4:]
    run(data_dir, split, ckpts).to_csv(out_csv)
    print("wrote", out_csv)

In [ ]:
# ---- Part B: our reader, rank-blended into Part A's submission with one weight per finding ----
import glob as _o_glob, os as _o_os, shutil as _o_shutil, subprocess as _o_sp, sys as _o_sys, time as _o_time
import pandas as _o_pd
OWN_W = 0.15
OWN_GLOB = "cnxt_v0_fold[012].pt"
OWN_MODE = "end"  # end: blend with the final submission; T / H: blend into one side of the stack's last blend
OWN_W_PER_TARGET = {"Baker's": 0.35, "Contusion": 0.3, "Medial OA": 0.25, "ACL": 0.25, "Medial Meniscus": 0.2, "MCL": 0.2, "Lateral Meniscus": 0.08, "Fracture": 0.08, "Lateral OA": 0.08, "PF OA": 0.08, "Effusion": 0.04, "Synovitis": 0.03}  # optional {finding: weight}; used with mode "end" when not empty
OWN_P33 = 0.0  # >0: weight of the public meniscus specialist (renta0426) inside the stack's Medial Meniscus rank
_o_pub, _o_bak = "/kaggle/working/submission.csv", "/kaggle/working/_public_stack.csv"
_o_shutil.copy(_o_pub, _o_bak)
try:
    def _o_find(name):
        for pat in ["/kaggle/input/*/", "/kaggle/input/*/*/", "/kaggle/input/*/*/*/"]:
            for d in sorted(_o_glob.glob(pat)):
                if _o_os.path.exists(_o_os.path.join(d, name)):
                    return d.rstrip("/")
        raise FileNotFoundError(name)
    _o_assets, _o_data = _o_find("knee.py"), _o_find("test_series.csv")
    _o_ck = sorted(c for g in OWN_GLOB.split() for c in _o_glob.glob(_o_assets + "/" + g))
    assert _o_ck, "no own checkpoints found"
    _o_env = dict(_o_os.environ)
    _o_target = "/kaggle/working/_own_env"
    _o_sp.run([_o_sys.executable, "-m", "pip", "install", "-q", "--no-index", "--no-deps", "--target", _o_target,
               "--find-links", _o_assets, "pylibjpeg", "pylibjpeg-libjpeg", "pylibjpeg-openjpeg", "pyjpegls", "python-gdcm"], check=False)
    _o_timm = [w for pat in ["/kaggle/input/*/timm-*.whl", "/kaggle/input/*/*/timm-*.whl", "/kaggle/input/*/*/*/timm-*.whl"]
               for w in _o_glob.glob(pat)]
    if _o_timm:
        _o_sp.run([_o_sys.executable, "-m", "pip", "install", "-q", "--no-deps", "--target", _o_target, _o_timm[0]], check=False)
    _o_env["PYTHONPATH"] = _o_target + _o_os.pathsep + _o_env.get("PYTHONPATH", "")
    _o_chk = ("import glob,os,numpy as np,pydicom;A=%r;ref=np.load(A+'/fixtures_ref.npy')\n"
             "for f in sorted(glob.glob(A+'/fixtures_*.dcm')):\n"
             "    try: print('decoder check',os.path.basename(f),'OK' if np.array_equal(pydicom.dcmread(f).pixel_array,ref) else 'MISMATCH',flush=True)\n"
             "    except Exception as e: print('decoder check',os.path.basename(f),'FAIL',repr(e)[:120],flush=True)\n") % _o_assets
    if _o_os.path.exists(_o_assets + "/fixtures_ref.npy"):  # compressed-DICOM decoders must work in the leg's environment
        _o_sp.run([_o_sys.executable, "-c", _o_chk], env=_o_env, check=False)
    try:  # run on the GPU with the most free memory
        import gc as _o_gc, torch as _o_torch
        _o_gc.collect(); _o_torch.cuda.empty_cache()
        _o_free = [_o_torch.cuda.mem_get_info(i)[0] for i in range(_o_torch.cuda.device_count())]
        _o_env["CUDA_VISIBLE_DEVICES"] = str(max(range(len(_o_free)), key=lambda i: _o_free[i]))
        print("own leg: free GPU memory (GB)", [round(f / 1e9, 1) for f in _o_free], flush=True)
    except Exception as _o_e:
        print("own leg: GPU pick skipped:", repr(_o_e))
    _o_t0 = _o_time.time()
    _o_sp.run([_o_sys.executable, "/kaggle/working/own_src/infer.py", _o_data, "test", "/kaggle/working/_own.csv"] + _o_ck,
              check=True, env=_o_env)
    _o_stack = _o_pd.read_csv(_o_bak, dtype={"StudyInstanceUID": str})
    if OWN_P33 > 0:
        try:  # rebuild the stack's Medial Meniscus rank with the specialist mixed into its transformer share
            _o_bag = _o_pd.read_csv("/kaggle/working/public0033_bag_raw.csv", dtype={"StudyInstanceUID": str})
            _o_tr = globals().get("_blend_tr", globals().get("_ke_tr"))
            _o_cr = globals().get("_blend_cr", globals().get("_ke_cr"))
            _o_uid = _ke_ours["StudyInstanceUID"].astype(str).tolist()
            _o_bagr = _o_bag.set_index("StudyInstanceUID")["Medial Meniscus"].reindex(_o_uid)
            assert _o_bagr.notna().all(), "specialist output does not cover every study"
            _o_bagr = _o_bagr.rank(method="average", pct=True).to_numpy()
            _o_wc = float(_coatnet_weight["Medial Meniscus"])
            _o_pre = (1 - _o_wc - OWN_P33) * _o_tr["Medial Meniscus"].to_numpy() + _o_wc * _o_cr["Medial Meniscus"].to_numpy() + OWN_P33 * _o_bagr
            _o_mm = _o_pd.Series(_o_pre, index=_o_uid).rank(method="average", pct=True)
            _o_old = _o_stack["Medial Meniscus"].copy()
            _o_stack["Medial Meniscus"] = _o_mm.reindex(_o_stack.StudyInstanceUID.astype(str)).to_numpy()
            assert _o_stack["Medial Meniscus"].notna().all(), "overlay produced NaN"
            print(f"own leg: meniscus specialist at {OWN_P33} in the stack's Medial Meniscus (transformer {1 - _o_wc - OWN_P33:.2f}, coat {_o_wc:.2f}); "
                  f"rank corr with the stack's column {_o_old.corr(_o_stack['Medial Meniscus'], method='spearman'):.3f}")
        except Exception as _o_pe:
            print("own leg: meniscus specialist overlay skipped:", repr(_o_pe))
    _o_own = _o_pd.read_csv("/kaggle/working/_own.csv", dtype={"StudyInstanceUID": str})
    _o_own = _o_own.set_index("StudyInstanceUID").loc[_o_stack.StudyInstanceUID].reset_index()
    _o_labels = [c for c in _o_stack.columns if c != "StudyInstanceUID"]
    assert _o_own[_o_labels].notna().all().all(), "own predictions contain NaN"
    assert (_o_own[_o_labels].nunique() > 1).all(), "own predictions are constant"
    _o_rank = lambda d: d[_o_labels].rank(method="average", pct=True)
    _o_final = _o_stack.copy()
    if OWN_MODE in ("T", "H") and "_ke_ours" in globals() and "_ke_theirs" in globals():
        # recompute the stack's last blend, rank((1-w)*rank(T) + w*H), with our leg mixed into one side
        _o_T = _ke_ours.set_index("StudyInstanceUID").loc[_o_stack.StudyInstanceUID]
        _o_H = _ke_theirs.set_index("StudyInstanceUID").loc[_o_stack.StudyInstanceUID]
        _o_side_T = _o_T[_o_labels].rank(method="average", pct=True).reset_index(drop=True)
        _o_side_H = _o_H[_o_labels].reset_index(drop=True)
        _o_ownr = _o_rank(_o_own).reset_index(drop=True)
        if OWN_MODE == "T":
            _o_side_T = ((1 - OWN_W) * _o_side_T + OWN_W * _o_ownr).rank(method="average", pct=True)
        else:
            _o_side_H = ((1 - OWN_W) * _o_side_H + OWN_W * _o_ownr).rank(method="average", pct=True)
        for _o_c in _o_labels:
            _o_w = float(_coatnet_weight[_o_c])
            _o_final[_o_c] = ((1 - _o_w) * _o_side_T[_o_c] + _o_w * _o_side_H[_o_c]).to_numpy()
        _o_final[_o_labels] = _o_final[_o_labels].rank(method="average", pct=True)
        print(f"own leg: mode {OWN_MODE}, mixed at {OWN_W} into the stack's {'transformer' if OWN_MODE == 'T' else 'raptor/coat'} side")
    else:
        _o_rs, _o_ro = _o_rank(_o_stack), _o_rank(_o_own)
        for _o_c in _o_labels:
            _o_w = float(OWN_W_PER_TARGET.get(_o_c, OWN_W)) if OWN_W_PER_TARGET else OWN_W
            _o_final[_o_c] = ((1 - _o_w) * _o_rs[_o_c] + _o_w * _o_ro[_o_c]).to_numpy()
        _o_final[_o_labels] = _o_final[_o_labels].rank(method="average", pct=True)
        if OWN_W_PER_TARGET:
            print("own leg: per-target weights", OWN_W_PER_TARGET)
    _o_final.to_csv(_o_pub, index=False)
    print(f"own leg: blended {len(_o_ck)} checkpoints at weight {OWN_W} in {_o_time.time() - _o_t0:.0f}s; "
          f"mean rank corr with stack {_o_rank(_o_stack).corrwith(_o_rank(_o_own)).mean():.3f}")
except Exception as _o_e:
    _o_shutil.copy(_o_bak, _o_pub)
    print("own leg FAILED, public stack submission kept:", repr(_o_e))

In [ ]:
# ---- Part C: our leg (R5t ConvNeXt-nano 336), rank blend at a fixed weight; any failure keeps the previous file ----
import gc, os, shutil, subprocess, sys
try:
    import torch; gc.collect(); torch.cuda.empty_cache()
except Exception as _e:
    print('cuda cleanup skipped', _e)
OURS_W = 0.1
OURS_TAGS = 'r5t_nano336,r5tfull_nano336'
_pub = '/kaggle/working/_ab_submission.csv'
shutil.copy('/kaggle/working/submission.csv', _pub)
_code = None
for _d, _dirs, _files in os.walk('/kaggle/input'):
    _dirs[:] = [x for x in _dirs if x not in ('competitions', 'train_series', 'test_series')]
    if os.path.basename(_d) == 'rsna_knee' and '__init__.py' in _files:
        _code = os.path.dirname(_d); break
print('our code root:', _code, flush=True)
_out = '/kaggle/working/_ours_submission.csv'
if _code and OURS_W > 0:
    _pp = [_code, '/kaggle/working/_own_env', os.environ.get('PYTHONPATH', '')]   # _own_env: Part B's DICOM decoders
    _env = {**os.environ, 'PYTHONPATH': os.pathsep.join(p for p in _pp if p)}
    _rc = subprocess.run([sys.executable, '-m', 'rsna_knee.mil.ours', '--public', _pub, '--out', _out,
                          '--w', str(OURS_W), '--tags', OURS_TAGS], env=_env).returncode
    print('ours leg exit code', _rc, flush=True)
    if _rc == 0 and os.path.exists(_out):
        import pandas as _pd
        _a, _b = _pd.read_csv(_pub), _pd.read_csv(_out)
        if list(_a.columns) == list(_b.columns) and len(_a) == len(_b) and _b.notna().all().all():
            shutil.copy(_out, '/kaggle/working/submission.csv')
            print('submission.csv = previous + ours blend at', OURS_W, flush=True)
